<a href="https://colab.research.google.com/github/ramahIO/Qirshak/blob/main/notebooks/Copy_of_qirshak_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# قِرْشَك | Qirshak — Smart Spending Watcher

مساعد مالي متعدد الوكلاء (CrewAI) يراقب صرفك، يكتشف الأنماط، وينبهك قبل ما تخسر السيطرة على ميزانيتك.

**بنية النوتبوك:**
1. الإعداد المشترك (الكل يشغّله أول)
2. المجموعة 1 — Categorizer + Pattern Detective
3. المجموعة 2 — Time-Based Alert + Predictor
4. المجموعة 3 — Budget & Savings Advisor + Tone Agent
5. تجميع الفريق الكامل (Crew) — يُكمّل بعد جهوزية كل المجموعات
6. التشغيل والاختبار النهائي

⚠️ كل عضو ينسخ القسم 1 كما هو بدون تعديل، ويشتغل على قسمه المخصص فقط.

# Section 1: Shared Setup

Everyone runs these four cells exactly as they are. Do not edit anything except your personal API key.

In [ ]:
# 1. Install
!pip install -q -U crewai pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.0/195.0 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4

In [ ]:
# 2. Imports
import os
import json
import pandas as pd

from datetime import datetime
from pydantic import BaseModel, Field

from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool

In [ ]:
# 3. Configure the LLM (each member enters their own key here at run time)
os.environ["OPENROUTER_API_KEY"] = ""

llm = LLM(
    model="openrouter/openai/gpt-4o-mini",
    base_url="https://openrouter.ai/api/v1",
    temperature=0
)

In [ ]:
# 4. Load the bank statement data (same file for everyone)

import urllib.request

url = "https://raw.githubusercontent.com/ramahIO/Qirshak/refs/heads/main/data/mock_statement.csv"
urllib.request.urlretrieve(url, "mock_statement.csv")

statement_df = pd.read_csv("mock_statement.csv")
statement_text = statement_df.to_string(index=False)

print("Statement loaded:", len(statement_df), "transactions")

Statement loaded: 43 transactions


In [ ]:
# التحقق الآلي من المجاميع (Ground Truth) — يُستخدم لاحقاً كمرجع تحقق
category_map = {
    "STARBUCKS COFFEE JEDDAH": "coffee", "BARNS COFFEE": "coffee", "DR CAFE COFFEE": "coffee",
    "JAHEZ DELIVERY": "delivery", "HUNGERSTATION": "delivery",
    "NETFLIX.COM": "subscriptions", "SHAHID VIP": "subscriptions", "SPOTIFY PREMIUM": "subscriptions",
    "UBER TRIP": "transport", "CAREEM RIDE": "transport",
    "PANDA HYPERMARKET": "groceries", "CARREFOUR MARKET": "groceries",
    "RENT PAYMENT TRANSFER": "bills_rent", "STC MOBILE BILL": "bills_rent",
    "ELECTRICITY BILL SEC": "bills_rent", "STC PAY TRANSFER": "bills_rent",
    "SALARY - COMPANY XYZ": "income"
}

statement_df["category_verified"] = statement_df["description"].map(category_map)
true_totals = statement_df.groupby("category_verified")["amount_sar"].sum().round(2)

print("VERIFIED TOTALS (calculated directly from data, not by the LLM):")
print(true_totals)

VERIFIED TOTALS (calculated directly from data, not by the LLM):
category_verified
bills_rent      -1900.00
coffee           -435.50
delivery         -348.00
groceries        -973.50
income           4000.00
subscriptions     -88.98
transport         -90.50
Name: amount_sar, dtype: float64


In [ ]:
verified_summary = f"""
VERIFIED MONTHLY SPENDING SUMMARY (calculated directly from data):

Income: {true_totals['income']:.2f} SAR (received on day 1 of the month)
Coffee: {true_totals['coffee']:.2f} SAR
Delivery: {true_totals['delivery']:.2f} SAR
Groceries: {true_totals['groceries']:.2f} SAR
Bills & Rent: {true_totals['bills_rent']:.2f} SAR
Subscriptions: {true_totals['subscriptions']:.2f} SAR
Transport: {true_totals['transport']:.2f} SAR

Total expenses: {true_totals.drop('income').sum():.2f} SAR
Remaining balance: {true_totals.sum():.2f} SAR

Statement covers a 30-day month. Last recorded transaction was on day 27.
No transactions were recorded for days 28-30 (balance nearly depleted).
"""

print(verified_summary)


VERIFIED MONTHLY SPENDING SUMMARY (calculated directly from data):

Income: 4000.00 SAR (received on day 1 of the month)
Coffee: -435.50 SAR
Delivery: -348.00 SAR
Groceries: -973.50 SAR
Bills & Rent: -1900.00 SAR
Subscriptions: -88.98 SAR
Transport: -90.50 SAR

Total expenses: -3836.48 SAR
Remaining balance: 163.52 SAR

Statement covers a 30-day month. Last recorded transaction was on day 27.
No transactions were recorded for days 28-30 (balance nearly depleted).



In [ ]:
print(statement_df.head(10))

         date   time              description  amount_sar category_verified
0  2026-07-01  09:00     SALARY - COMPANY XYZ     4000.00            income
1  2026-07-01  17:15  STARBUCKS COFFEE JEDDAH      -22.00            coffee
2  2026-07-02  08:30        PANDA HYPERMARKET     -186.50         groceries
3  2026-07-02  18:00           JAHEZ DELIVERY      -54.00          delivery
4  2026-07-03  11:00          STC MOBILE BILL     -120.00        bills_rent
5  2026-07-03  16:45             BARNS COFFEE      -19.00            coffee
6  2026-07-03  20:10                UBER TRIP      -28.50         transport
7  2026-07-04  17:30           DR CAFE COFFEE      -24.00            coffee
8  2026-07-05  09:00    RENT PAYMENT TRANSFER    -1500.00        bills_rent
9  2026-07-05  10:00              NETFLIX.COM      -39.99     subscriptions


---
# Section 2: Group 1 — Categorizer + Pattern Detective

**Receives:** `statement_text` directly from Section 1

**Delivers:** a text report with (a) every transaction categorized, (b) a list of detected patterns

In [ ]:
from crewai.tools import tool

@tool("Category Total Calculator")
def calculate_category_total(amounts: str) -> str:
    """Takes a comma-separated list of numbers (e.g. '22.00,19.00,24.00')
    and returns their exact sum, calculated in Python for guaranteed
    accuracy. Use this whenever you need to add up transaction amounts
    for a category — never add them manually."""
    numbers = [float(x.strip()) for x in amounts.split(",")]
    total = sum(numbers)
    return f"{total:.2f}"

In [ ]:
categorizer = Agent(
    role="Personal Spending Categorizer",

    goal=(
        "Classify every transaction in a bank statement into one clear "
        "category, using only the transaction description and amount "
        "provided — never guessing beyond what the data shows."
    ),

    backstory=(
        "You are a meticulous financial analyst who reads raw bank "
        "statement rows and assigns each one to exactly one category: "
        "coffee, delivery, subscriptions, transport, groceries, "
        "bills_rent, income, or other. You never invent a transaction "
        "that isn't in the data, and you never assign two categories "
        "to the same row. You NEVER add numbers manually in your head — "
        "whenever you need a total for a category, you MUST call the "
        "Category Total Calculator tool with the list of amounts."
    ),

    tools=[calculate_category_total],
    llm=llm,
    verbose=True,
    allow_delegation=False
)

In [ ]:
# --- Agent 2: Pattern Detective ---
pattern_detective = Agent(
    role="Spending Pattern Detective",

    goal=(
        "Analyze categorized spending data to surface behavioral patterns "
        "the person may not have noticed themselves — repetition, "
        "overlapping subscriptions, and timing trends — based only on "
        "the data provided."
    ),

    backstory=(
        "You are a behavioral finance analyst who specializes in finding "
        "quiet spending leaks: small repeated purchases that add up, "
        "subscriptions that overlap in purpose, and differences between "
        "weekday and weekend spending. You report only patterns that are "
        "clearly supported by the data — you never claim a subscription "
        "is 'unused' unless the data actually shows that, since usage "
        "isn't something a bank statement can confirm."
    ),

    llm=llm,
    verbose=True,
    allow_delegation=False
)

In [ ]:
# --- Task 1: Categorization Task ---
categorization_task = Task(
    description=f'''
Categorize every transaction in the bank statement below into exactly
one of these categories: coffee, delivery, subscriptions, transport,
groceries, bills_rent, income, other.

{statement_text}

Rules:

- Use only the transaction description and amount provided.
- Do not invent transactions that are not in the data.
- Assign exactly one category per transaction — never two.
- Treat any positive amount as income.
- After grouping transactions by category, you MUST use the Category
  Total Calculator tool to compute the total for each category — do
  not add the numbers yourself.
- Group your output by category, and for each category state:
  the transactions in it, the count, and the total amount (from the tool).
- Do not skip any transaction from the statement.
''',
    expected_output=(
        "A structured report grouping every transaction into its "
        "category, with a tool-calculated total (SAR) and transaction "
        "count for each category."
    ),
    agent=categorizer
)

In [ ]:
# --- Task 2: Pattern Detection Task ---
pattern_task = Task(
    description='''
Based on the categorized spending report above, identify behavioral
spending patterns.

Look specifically for:

1. Repeated small purchases in the same category (e.g. frequent coffee
   purchases) — note the frequency and typical time of day if visible
   in the data.
2. Subscriptions or recurring charges that fall into the same category
   (e.g. multiple streaming services) — flag them as worth reviewing,
   without claiming any of them are unused.
3. Any noticeable difference between spending on different days of the
   week, if the data shows one.

Rules:

- Base every pattern strictly on the categorized data — do not invent
  a pattern the data does not support.
- Do not make claims about whether a subscription is used or unused.
- If a pattern is weak or uncertain, say so explicitly rather than
  presenting it as a strong finding.
''',
    expected_output=(
        "A short list of clearly supported spending patterns, each with "
        "the evidence behind it (frequency, amount, or timing), and any "
        "patterns explicitly flagged as uncertain."
    ),
    agent=pattern_detective,
    context=[categorization_task]
)

In [ ]:
# --- Test Group 1 on its own (mini Crew, no need to wait for other groups) ---
group1_crew = Crew(
    agents=[categorizer, pattern_detective],
    tasks=[categorization_task, pattern_task],
    process=Process.sequential,
    verbose=True
)

group1_result = await group1_crew.kickoff_async()
print(group1_result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 9edcaf36-988b-4914-aefe-422ce81ade5b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Categorize every transaction in the bank statement below into exactly                                          │
│  one of these categories: coffee, delivery, subscriptions, transport,                                           │
│  groceries, bills_rent, income, other.                                                                          │
│                                                                                                                 │
│        date  time             description  amount_sar                                                           │
│  2026-07-01 09:00    SALARY - COMPANY XYZ     4000.00                                                           │
│  2026-07-01 17:15 STARBUCKS COFFEE JEDDAH      -22.00                                                           │
│  2026-07-02 08:30       PANDA HYPERMARKET     -186.50                                                           │
│  2026-07-02 18:00          JAHEZ DELIVERY      -54.00                                                           │
│  2026-07-03 11:00         STC MOBILE BILL     -120.00                                                           │
│  2026-07-03 16:45            BARNS COFFEE      -19.00                                                           │
│  2026-07-03 20:10               UBER TRIP      -28.50                                                           │
│  2026-07-04 17:30          DR CAFE COFFEE      -24.00                                                           │
│  2026-07-05 09:00   RENT PAYMENT TRANSFER    -1500.00                                                           │
│  2026-07-05 10:00             NETFLIX.COM      -39.99                                                           │
│  2026-07-05 10:02              SHAHID VIP      -29.00                                                           │
│  2026-07-05 10:05         SPOTIFY PREMIUM      -19.99                                                           │
│  2026-07-05 16:50 STARBUCKS COFFEE JEDDAH      -21.00                                                           │
│  2026-07-06 12:20        CARREFOUR MARKET     -142.00                                                           │
│  2026-07-06 19:40           HUNGERSTATION      -63.00                                                           │
│  2026-07-07 17:10            BARNS COFFEE      -18.50                                                           │
│  2026-07-08 16:40 STARBUCKS COFFEE JEDDAH      -23.00                                                           │
│  2026-07-08 21:00             CAREEM RIDE      -32.00                                                           │
│  2026-07-09 17:55          DR CAFE COFFEE      -20.00                                                           │
│  2026-07-10 13:15       PANDA HYPERMARKET     -210.00                                                           │
│  2026-07-10 18:20          JAHEZ DELIVERY      -48.00                                                           │
│  2026-07-11 16:35 STARBUCKS COFFEE JEDDAH      -25.00                                                           │
│  2026-07-12 10:00    ELECTRICITY BILL SEC     -180.00                                                           │
│  2026-07-12 17:05            BARNS COFFEE      -19.50                                                           │
│  2026-07-13 09:45        STC PAY TRANSFER     -100.00                                                           │
│  2026-07-13 18:00           HUNGERSTATION      -58.00  

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Personal Spending Categorizer                                                                           │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Categorize every transaction in the bank statement below into exactly                                          │
│  one of these categories: coffee, delivery, subscriptions, transport,                                           │
│  groceries, bills_rent, income, other.                                                                          │
│                                                                                                                 │
│        date  time             description  amount_sar                                                           │
│  2026-07-01 09:00    SALARY - COMPANY XYZ     4000.00                                                           │
│  2026-07-01 17:15 STARBUCKS COFFEE JEDDAH      -22.00                                                           │
│  2026-07-02 08:30       PANDA HYPERMARKET     -186.50                                                           │
│  2026-07-02 18:00          JAHEZ DELIVERY      -54.00                                                           │
│  2026-07-03 11:00         STC MOBILE BILL     -120.00                                                           │
│  2026-07-03 16:45            BARNS COFFEE      -19.00                                                           │
│  2026-07-03 20:10               UBER TRIP      -28.50                                                           │
│  2026-07-04 17:30          DR CAFE COFFEE      -24.00                                                           │
│  2026-07-05 09:00   RENT PAYMENT TRANSFER    -1500.00                                                           │
│  2026-07-05 10:00             NETFLIX.COM      -39.99                                                           │
│  2026-07-05 10:02              SHAHID VIP      -29.00                                                           │
│  2026-07-05 10:05         SPOTIFY PREMIUM      -19.99                                                           │
│  2026-07-05 16:50 STARBUCKS COFFEE JEDDAH      -21.00                                                           │
│  2026-07-06 12:20        CARREFOUR MARKET     -142.00                                                           │
│  2026-07-06 19:40           HUNGERSTATION      -63.00                                                           │
│  2026-07-07 17:10            BARNS COFFEE      -18.50                                                           │
│  2026-07-08 16:40 STARBUCKS COFFEE JEDDAH      -23.00                                                           │
│  2026-07-08 21:00             CAREEM RIDE      -32.00                                                           │
│  2026-07-09 17:55          DR CAFE COFFEE      -20.00                                                           │
│  2026-07-10 13:15       PANDA HYPERMARKET     -210.00                                                           │
│  2026-07-10 18:20          JAHEZ DELIVERY      -48.00                                                           │
│  2026-07-11 16:35 STARBUCKS COFFEE JEDDAH      -25.00                                                           │
│  2026-07-12 10:00    ELECTRICITY BILL SEC     -180.00                                                           │
│  2026-07-12 17:05            BARNS COFFEE      -19.50                                                           │
│  2026-07-13 09:45        STC PAY TRANSFER     -100.00  

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: category_total_calculator                                                                                │
│  Args: {'amounts': '4000.00'}                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: category_total_calculator                                                                                │
│  Args: {'amounts':                                                                                              │
│  '-22.00,-19.00,-24.00,-21.00,-18.50,-23.00,-20.00,-24.00,-22.50,-26.00,-22.50,-23.50,-20.50,-24.50,-22.00'}    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Spending Pattern Detective                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Spending Patterns Identified                                                                               │
│                                                                                                                 │
│  1. **Repeated Small Purchases in Coffee Category:**                                                            │
│     - **Frequency:** 20 transactions                                                                            │
│     - **Total Amount:** -333.00 SAR                                                                             │
│     - **Typical Time of Day:** Not specified in the data.                                                       │
│     - **Details:** The individual has made numerous coffee purchases across three different coffee shops        │
│  (Starbucks, Barns, and Dr. Cafe). This indicates a consistent habit of buying coffee, which could be a         │
│  potential area for cost-saving if desired.                                                                     │
│                                                                                                                 │
│  2. **Subscriptions Worth Reviewing:**                                                                          │
│     - **Count:** 3 subscriptions                                                                                │
│     - **Total Amount:** -88.98 SAR                                                                              │
│     - **Subscriptions:**                                                                                        │
│       - Netflix.com (-39.99 SAR)                                                                                │
│       - Shahid VIP (-29.00 SAR)                                                                                 │
│       - Spotify Premium (-19.99 SAR)                                                                            │
│     - **Details:** These subscriptions fall into the entertainment category. While they serve different         │
│  purposes (streaming video vs. music), it may be worth reviewing if there is any overlap in content or usage.   │
│                                                                                                                 │
│  3. **Delivery Spending:**                                                                                      │
│     - **Frequency:** 5 transactions                                                                             │
│     - **Total Amount:** -293.00 SAR                                                                             │
│     - **Details:** The individual has made multiple purchases through delivery services (Jahez and              │
│  HungerStation). This could indicate a preference for convenience, but it may also be an area to evaluate for   │
│  potential savings.                                                                                             │
│                                                                                                                 │
│  4. **Grocery Spending:**                                                                                       │
│     - **Frequency:** 5 transactions                                                                             │
│     - **Total Amount:** -808.50 SAR                    

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Based on the categorized spending report above, identify behavioral                                            │
│  spending patterns.                                                                                             │
│                                                                                                                 │
│  Look specifically for:                                                                                         │
│                                                                                                                 │
│  1. Repeated small purchases in the same category (e.g. frequent coffee                                         │
│     purchases) — note the frequency and typical time of day if visible                                          │
│     in the data.                                                                                                │
│  2. Subscriptions or recurring charges that fall into the same category                                         │
│     (e.g. multiple streaming services) — flag them as worth reviewing,                                          │
│     without claiming any of them are unused.                                                                    │
│  3. Any noticeable difference between spending on different days of the                                         │
│     week, if the data shows one.                                                                                │
│                                                                                                                 │
│  Rules:                                                                                                         │
│                                                                                                                 │
│  - Base every pattern strictly on the categorized data — do not invent                                          │
│    a pattern the data does not support.                                                                         │
│  - Do not make claims about whether a subscription is used or unused.                                           │
│  - If a pattern is weak or uncertain, say so explicitly rather than                                             │
│    presenting it as a strong finding.                                                                           │
│                                                                                                                 │
│  Agent: Spending Pattern Detective                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Spending Patterns Identified

1. **Repeated Small Purchases in Coffee Category:**
   - **Frequency:** 20 transactions
   - **Total Amount:** -333.00 SAR
   - **Typical Time of Day:** Not specified in the data.
   - **Details:** The individual has made numerous coffee purchases across three different coffee shops (Starbucks, Barns, and Dr. Cafe). This indicates a consistent habit of buying coffee, which could be a potential area for cost-saving if desired.

2. **Subscriptions Worth Reviewing:**
   - **Count:** 3 subscriptions
   - **Total Amount:** -88.98 SAR
   - **Subscriptions:**
     - Netflix.com (-39.99 SAR)
     - Shahid VIP (-29.00 SAR)
     - Spotify Premium (-19.99 SAR)
   - **Details:** These subscriptions fall into the entertainment category. While they serve different purposes (streaming video vs. music), it may be worth reviewing if there is any overlap in content or usage.

3. **Delivery Spending:**
   - **Frequency:** 5 transactions
   - **Total Amount:** -293.00 SAR

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 9edcaf36-988b-4914-aefe-422ce81ade5b                                                                       │
│  Final Output: ### Spending Patterns Identified                                                                 │
│                                                                                                                 │
│  1. **Repeated Small Purchases in Coffee Category:**                                                            │
│     - **Frequency:** 20 transactions                                                                            │
│     - **Total Amount:** -333.00 SAR                                                                             │
│     - **Typical Time of Day:** Not specified in the data.                                                       │
│     - **Details:** The individual has made numerous coffee purchases across three different coffee shops        │
│  (Starbucks, Barns, and Dr. Cafe). This indicates a consistent habit of buying coffee, which could be a         │
│  potential area for cost-saving if desired.                                                                     │
│                                                                                                                 │
│  2. **Subscriptions Worth Reviewing:**                                                                          │
│     - **Count:** 3 subscriptions                                                                                │
│     - **Total Amount:** -88.98 SAR                                                                              │
│     - **Subscriptions:**                                                                                        │
│       - Netflix.com (-39.99 SAR)                                                                                │
│       - Shahid VIP (-29.00 SAR)                                                                                 │
│       - Spotify Premium (-19.99 SAR)                                                                            │
│     - **Details:** These subscriptions fall into the entertainment category. While they serve different         │
│  purposes (streaming video vs. music), it may be worth reviewing if there is any overlap in content or usage.   │
│                                                                                                                 │
│  3. **Delivery Spending:**                                                                                      │
│     - **Frequency:** 5 transactions                                                                             │
│     - **Total Amount:** -293.00 SAR                                                                             │
│     - **Details:** The individual has made multiple purchases through delivery services (Jahez and              │
│  HungerStation). This could indicate a preference for convenience, but it may also be an area to evaluate for   │
│  potential savings.                                                                                             │
│                                                                                                                 │
│  4. **Grocery Spending:**                                                                                       │
│     - **Frequency:** 5 transactions                                                                             │
│     - **Total Amount:** -808.50 SAR                   

---
# Section 3: Group 2 — Time-Based Alert + Predictor

**Receives:** Group 1's output (or a temporary placeholder for early independent testing)

**Delivers:** (a) a live alert if triggered, (b) a projection of when the balance will run out before month end

In [ ]:
# Temporary test data until Group 1 is completed

simulated_current_time = "5:30 PM"

alert_confidence_threshold = 0.75
spending_increase_threshold = 30

placeholder_categorized_data = {
    "spending_patterns": [
        {
            "category": "Coffee",
            "transaction_count": 15,
            "average_transaction_amount": 22,
            "usual_time_range": "4:00 PM - 7:00 PM",
            "today_spending": 60,
            "usual_daily_spending": 35,
            "pattern_confidence": 0.88
        },
        {
            "category": "Food Delivery",
            "transaction_count": 8,
            "average_transaction_amount": 45,
            "usual_time_range": "7:00 PM - 10:00 PM",
            "today_spending": 40,
            "usual_daily_spending": 45,
            "pattern_confidence": 0.72
        }
    ],
    "current_balance": 1200,
    "total_spent_this_month": 1800,
    "days_elapsed": 15,
    "days_remaining": 15,
    "days_in_month": 30
}

print("Simulated current time:", simulated_current_time)
print("Confidence threshold:", alert_confidence_threshold)
print("Spending increase threshold:", spending_increase_threshold, "%")
print(json.dumps(placeholder_categorized_data, indent=2))

Simulated current time:

 5:30 PM
Confidence threshold: 0.75
Spending increase threshold: 30 %
{
  "spending_patterns": [
    {
      "category": "Coffee",
      "transaction_count": 15,
      "average_transaction_amount": 22,
      "usual_time_range": "4:00 PM - 7:00 PM",
      "today_spending": 60,
      "usual_daily_spending": 35,
      "pattern_confidence": 0.88
    },
    {
      "category": "Food Delivery",
      "transaction_count": 8,
      "average_transaction_amount": 45,
      "usual_time_range": "7:00 PM - 10:00 PM",
      "today_spending": 40,
      "usual_daily_spending": 45,
      "pattern_confidence": 0.72
    }
  ],
  "current_balance": 1200,
  "total_spent_this_month": 1800,
  "days_elapsed": 15,
  "days_remaining": 15,
  "days_in_month": 30
}
╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                               

In [ ]:
from datetime import datetime
import json

@tool("Time-Based Spending Checker")
def time_based_spending_checker(
    current_time: str,
    usual_time_range: str,
    today_spending: float,
    usual_daily_spending: float,
    pattern_confidence: float,
    confidence_threshold: float,
    increase_threshold: float
) -> str:
    """
    Checks whether the current time is within the user's usual spending period
    and whether today's spending is unusually high.
    """

    current = datetime.strptime(current_time, "%I:%M %p")

    start_text, end_text = [
        item.strip()
        for item in usual_time_range.split("-")
    ]

    start_time = datetime.strptime(start_text, "%I:%M %p")
    end_time = datetime.strptime(end_text, "%I:%M %p")

    is_within_usual_time = start_time <= current <= end_time

    if usual_daily_spending > 0:
        increase_percentage = (
            (today_spending - usual_daily_spending)
            / usual_daily_spending
        ) * 100
    else:
        increase_percentage = 0

    spending_is_high = increase_percentage >= increase_threshold
    confidence_is_enough = pattern_confidence >= confidence_threshold

    alert_required = (
        is_within_usual_time
        and spending_is_high
        and confidence_is_enough
    )

    result = {
        "alert_required": alert_required,
        "is_within_usual_time": is_within_usual_time,
        "spending_is_high": spending_is_high,
        "confidence_is_enough": confidence_is_enough,
        "today_spending": today_spending,
        "usual_daily_spending": usual_daily_spending,
        "increase_percentage": round(increase_percentage, 2),
        "pattern_confidence": pattern_confidence,
        "confidence_threshold": confidence_threshold,
        "increase_threshold": increase_threshold
    }

    return json.dumps(result)

In [ ]:
# --- Agent 3: Time-Based Alert Agent ---
# Agent 3: Time-Based Alert Agent ---
time_alert_agent = Agent(
    role="Time-Based Spending Alert Specialist",

    goal=(
        "Analyze the user's spending behavior based on time patterns, "
        "compare the current spending with historical spending habits, "
        "and generate a spending alert only when unusual behavior is detected."
    ),

    backstory=(
        "You are a financial behavior specialist in the Qirshak system. "
        "You analyze spending habits based on historical transaction patterns. "
        "You compare the current time with the user's usual spending period, "
        "evaluate today's spending against normal spending, and issue alerts only "
        "when the evidence strongly indicates unusual financial behavior. "
        "Never invent data or modify financial values."
    ),

    tools=[time_based_spending_checker],

    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=3
)

In [ ]:
@tool("Balance Depletion Predictor")
def balance_depletion_predictor(
    current_balance: float,
    total_spent_this_month: float,
    days_elapsed: int,
    days_remaining: int
) -> str:
    """
    Calculates the user's average daily spending and predicts whether
    the remaining balance will run out before the end of the month.
    """

    if days_elapsed <= 0:
        return json.dumps({
            "error": "days_elapsed must be greater than zero"
        })

    average_daily_spending = total_spent_this_month / days_elapsed

    if average_daily_spending <= 0:
        estimated_days_until_depletion = None
        will_run_out_early = False
        days_early = 0
    else:
        estimated_days_until_depletion = (
            current_balance / average_daily_spending
        )

        will_run_out_early = (
            estimated_days_until_depletion < days_remaining
        )

        if will_run_out_early:
            days_early = (
                days_remaining - estimated_days_until_depletion
            )
        else:
            days_early = 0

    result = {
        "current_balance": current_balance,
        "total_spent_this_month": total_spent_this_month,
        "days_elapsed": days_elapsed,
        "days_remaining": days_remaining,
        "average_daily_spending": round(average_daily_spending, 2),
        "estimated_days_until_depletion": (
            round(estimated_days_until_depletion, 2)
            if estimated_days_until_depletion is not None
            else None
        ),
        "will_run_out_early": will_run_out_early,
        "days_early": round(days_early, 2)
    }

    return json.dumps(result)

In [ ]:
# --- Agent 4: Predictor ---
predictor = Agent(
    role="Spending Predictor",

    goal=(
        "Use verified financial data and the Balance Depletion Predictor tool "
        "to estimate whether the user's remaining balance will run out before "
        "the end of the month, and determine approximately how many days early."
    ),

    backstory=(
        "You are a careful financial forecaster in the Qirshak system. "
        "You rely only on verified financial values and the results returned "
        "by the Balance Depletion Predictor tool. "
        "You never invent, modify, or recalculate financial figures outside "
        "the provided tool results. Your responsibility is to interpret the "
        "forecast clearly and explain the financial risk without exaggeration."
    ),

    tools=[balance_depletion_predictor],

    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=3
)

In [ ]:
# ---------- Short-Term Session Memory (خاصة بـ Time-Based Alert Agent فقط) ----------
# يطابق البنية المعمارية الأصلية بالمقترح:
# "Agent 3 uses short-term memory to hold today's running spend-so-far as session state"

session_memory = {
    "today_coffee_spending": 0.0,
    "transactions_today": []
}

def add_transaction(amount: float, description: str = "Coffee"):
    """تسجل عملية شراء جديدة بذاكرة الجلسة — تتراكم طول ما الجلسة شغالة."""
    session_memory["today_coffee_spending"] += amount
    session_memory["transactions_today"].append(description)
    return session_memory["today_coffee_spending"]

In [ ]:
# محاكاة: المستخدم اشترى 3 قهوات اليوم — كل عملية تُسجل وتتراكم بالذاكرة
add_transaction(24, "Starbucks - Coffee")
add_transaction(20, "Barn's - Coffee")
add_transaction(22, "Dr. Cafe - Coffee")

print("إجمالي صرف اليوم (من الذاكرة):", session_memory["today_coffee_spending"], "ريال")
print("عدد العمليات المسجلة اليوم:", session_memory["transactions_today"])

إجمالي صرف اليوم (من الذاكرة): 66.0 ريال
عدد العمليات المسجلة اليوم: ['Starbucks - Coffee', "Barn's - Coffee", 'Dr. Cafe - Coffee']


In [ ]:
# البيانات الحقيقية لوكيل التنبيه اللحظي (بدل placeholder_categorized_data الوهمي)
real_coffee_pattern = {
    "spending_patterns": [
        {
            "category": "Coffee",
            "transaction_count": 20,
            "average_transaction_amount": round(435.50 / 20, 2),
            "usual_time_range": "4:00 PM - 8:00 PM",
            "today_spending": session_memory["today_coffee_spending"],
            "usual_daily_spending": round(435.50 / 20, 2),
            "pattern_confidence": 0.88
        }
    ],
    "current_balance": 163.52,
    "total_spent_this_month": 3836.48,
    "days_elapsed": 27,
    "days_remaining": 3,
    "days_in_month": 30
}

print("REAL COFFEE PATTERN FOR ALERT AGENT:")
print(json.dumps(real_coffee_pattern, indent=2))

REAL COFFEE PATTERN FOR ALERT AGENT:
{
  "spending_patterns": [
    {
      "category": "Coffee",
      "transaction_count": 20,
      "average_transaction_amount": 21.77,
      "usual_time_range": "4:00 PM - 8:00 PM",
      "today_spending": 66.0,
      "usual_daily_spending": 21.77,
      "pattern_confidence": 0.88
    }
  ],
  "current_balance": 163.52,
  "total_spent_this_month": 3836.48,
  "days_elapsed": 27,
  "days_remaining": 3,
  "days_in_month": 30
}


In [ ]:
# --- Task 3: Time-Based Alert Task ---
alert_task = Task(
    description=f"""
You are given the user's spending pattern analysis.

Current simulated time:
{simulated_current_time}

Spending pattern data:
{real_coffee_pattern}

Your task is to:

1. Compare the current simulated time with the user's usual spending time.

2. Compare today's spending with the user's usual daily spending.

3. Use the confidence threshold ({alert_confidence_threshold}) before making any decision.

4. Use the spending increase threshold ({spending_increase_threshold}%).

5. Use the Time-Based Spending Checker tool to analyze the data.

6. Decide whether a spending alert should be generated.

7. Explain the reason for your decision.
""",

    expected_output="""
Return a JSON object containing:

- alert_required
- reason
- current_time
- usual_time_range
- increase_percentage
- confidence_score
""",

    agent=time_alert_agent
)

In [ ]:
alert_test_crew = Crew(
    agents=[time_alert_agent],
    tasks=[alert_task],
    process=Process.sequential,
    verbose=True
)

alert_result = await alert_test_crew.kickoff_async()
print(alert_result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: a50f8e23-c374-4ba2-98f2-b80178d543d2                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  You are given the user's spending pattern analysis.                                                            │
│                                                                                                                 │
│  Current simulated time:                                                                                        │
│  5:30 PM                                                                                                        │
│                                                                                                                 │
│  Spending pattern data:                                                                                         │
│  {'spending_patterns': [{'category': 'Coffee', 'transaction_count': 20, 'average_transaction_amount': 21.77,    │
│  'usual_time_range': '4:00 PM - 8:00 PM', 'today_spending': 66, 'usual_daily_spending': 21.77,                  │
│  'pattern_confidence': 0.88}], 'current_balance': 163.52, 'total_spent_this_month': 3836.48, 'days_elapsed':    │
│  27, 'days_remaining': 3, 'days_in_month': 30}                                                                  │
│                                                                                                                 │
│  Your task is to:                                                                                               │
│                                                                                                                 │
│  1. Compare the current simulated time with the user's usual spending time.                                     │
│                                                                                                                 │
│  2. Compare today's spending with the user's usual daily spending.                                              │
│                                                                                                                 │
│  3. Use the confidence threshold (0.75) before making any decision.                                             │
│                                                                                                                 │
│  4. Use the spending increase threshold (30%).                                                                  │
│                                                                                                                 │
│  5. Use the Time-Based Spending Checker tool to analyze the data.                                               │
│                                                                                                                 │
│  6. Decide whether a spending alert should be generated.                                                        │
│                                                                                                                 │
│  7. Explain the reason for your decision.                                                                       │
│                                                                                                                 │
│  ID: 709e1445-c361-44cd-917f-c5fd0a41a06f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰────────────────────────────────────────────────────────

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Time-Based Spending Alert Specialist                                                                    │
│                                                                                                                 │
│  Task:                                                                                                          │
│  You are given the user's spending pattern analysis.                                                            │
│                                                                                                                 │
│  Current simulated time:                                                                                        │
│  5:30 PM                                                                                                        │
│                                                                                                                 │
│  Spending pattern data:                                                                                         │
│  {'spending_patterns': [{'category': 'Coffee', 'transaction_count': 20, 'average_transaction_amount': 21.77,    │
│  'usual_time_range': '4:00 PM - 8:00 PM', 'today_spending': 66, 'usual_daily_spending': 21.77,                  │
│  'pattern_confidence': 0.88}], 'current_balance': 163.52, 'total_spent_this_month': 3836.48, 'days_elapsed':    │
│  27, 'days_remaining': 3, 'days_in_month': 30}                                                                  │
│                                                                                                                 │
│  Your task is to:                                                                                               │
│                                                                                                                 │
│  1. Compare the current simulated time with the user's usual spending time.                                     │
│                                                                                                                 │
│  2. Compare today's spending with the user's usual daily spending.                                              │
│                                                                                                                 │
│  3. Use the confidence threshold (0.75) before making any decision.                                             │
│                                                                                                                 │
│  4. Use the spending increase threshold (30%).                                                                  │
│                                                                                                                 │
│  5. Use the Time-Based Spending Checker tool to analyze the data.                                               │
│                                                                                                                 │
│  6. Decide whether a spending alert should be generated.                                                        │
│                                                                                                                 │
│  7. Explain the reason for your decision.                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool time_based_spending_checker executed with result: {"alert_required": true, "is_within_usual_time": true, "spending_is_high": true, "confidence_is_enough": true, "today_spending": 66.0, "usual_daily_spending": 21.77, "increase_percentage": 203.17, "pa...

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: time_based_spending_checker                                                                              │
│  Args: {'current_time': '5:30 PM', 'usual_time_range': '4:00 PM - 8:00 PM', 'today_spending': 66,               │
│  'usual_daily_spending': 21.77, 'pattern_confidence': 0.88, 'confidence_threshold': 0.75,                       │
│  'increase_threshold'...                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Time-Based Spending Alert Specialist                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```json                                                                                                        │
│  {                                                                                                              │
│    "alert_required": true,                                                                                      │
│    "reason": "Today's spending of $66 is significantly higher than the usual daily spending of $21.77, with an  │
│  increase percentage of 203.17%. This is well above the 30% increase threshold, and the spending is occurring   │
│  within the user's usual spending time range.",                                                                 │
│    "current_time": "5:30 PM",                                                                                   │
│    "usual_time_range": "4:00 PM - 8:00 PM",                                                                     │
│    "increase_percentage": 203.17,                                                                               │
│    "confidence_score": 0.88                                                                                     │
│  }                                                                                                              │
│  ```                                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  You are given the user's spending pattern analysis.                                                            │
│                                                                                                                 │
│  Current simulated time:                                                                                        │
│  5:30 PM                                                                                                        │
│                                                                                                                 │
│  Spending pattern data:                                                                                         │
│  {'spending_patterns': [{'category': 'Coffee', 'transaction_count': 20, 'average_transaction_amount': 21.77,    │
│  'usual_time_range': '4:00 PM - 8:00 PM', 'today_spending': 66, 'usual_daily_spending': 21.77,                  │
│  'pattern_confidence': 0.88}], 'current_balance': 163.52, 'total_spent_this_month': 3836.48, 'days_elapsed':    │
│  27, 'days_remaining': 3, 'days_in_month': 30}                                                                  │
│                                                                                                                 │
│  Your task is to:                                                                                               │
│                                                                                                                 │
│  1. Compare the current simulated time with the user's usual spending time.                                     │
│                                                                                                                 │
│  2. Compare today's spending with the user's usual daily spending.                                              │
│                                                                                                                 │
│  3. Use the confidence threshold (0.75) before making any decision.                                             │
│                                                                                                                 │
│  4. Use the spending increase threshold (30%).                                                                  │
│                                                                                                                 │
│  5. Use the Time-Based Spending Checker tool to analyze the data.                                               │
│                                                                                                                 │
│  6. Decide whether a spending alert should be generated.                                                        │
│                                                                                                                 │
│  7. Explain the reason for your decision.                                                                       │
│                                                                                                                 │
│  Agent: Time-Based Spending Alert Specialist                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰────────────────────────────────────────────────────────

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: a50f8e23-c374-4ba2-98f2-b80178d543d2                                                                       │
│  Final Output: ```json                                                                                          │
│  {                                                                                                              │
│    "alert_required": true,                                                                                      │
│    "reason": "Today's spending of $66 is significantly higher than the usual daily spending of $21.77, with an  │
│  increase percentage of 203.17%. This is well above the 30% increase threshold, and the spending is occurring   │
│  within the user's usual spending time range.",                                                                 │
│    "current_time": "5:30 PM",                                                                                   │
│    "usual_time_range": "4:00 PM - 8:00 PM",                                                                     │
│    "increase_percentage": 203.17,                                                                               │
│    "confidence_score": 0.88                                                                                     │
│  }                                                                                                              │
│  ```                                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

```json
{
  "alert_required": true,
  "reason": "Today's spending of $66 is significantly higher than the usual daily spending of $21.77, with an increase percentage of 203.17%. This is well above the 30% increase threshold, and the spending is occurring within the user's usual spending time range.",
  "current_time": "5:30 PM",
  "usual_time_range": "4:00 PM - 8:00 PM",
  "increase_percentage": 203.17,
  "confidence_score": 0.88
}
```


In [ ]:
# البيانات الحقيقية للـ Predictor (بدل placeholder_categorized_data الوهمي)
real_financial_data = {
    "current_balance": round(true_totals.sum(), 2),
    "total_spent_this_month": round(abs(true_totals.drop('income').sum()), 2),
    "days_elapsed": 27,
    "days_remaining": 3
}

print("REAL FINANCIAL DATA FOR PREDICTOR:")
print(real_financial_data)


REAL FINANCIAL DATA FOR PREDICTOR:
{'current_balance': np.float64(163.52), 'total_spent_this_month': np.float64(3836.48), 'days_elapsed': 27, 'days_remaining': 3}


In [ ]:
# --- Task 4: Prediction Task ---
prediction_task = Task(
    description=f"""
You are given verified financial data for the user.

Financial data:
{real_financial_data}

Your task is to:

1. Use the Balance Depletion Predictor tool.

2. Pass these exact values to the tool:
   - current_balance
   - total_spent_this_month
   - days_elapsed
   - days_remaining

3. Use only the values returned by the tool.

4. Explain:
   - the average daily spending rate
   - how many days the remaining balance can cover
   - whether the balance will run out before month-end
   - approximately how many days early it may run out

5. Do not invent, modify, or independently recalculate any financial value.

6. Clearly state that the prediction assumes the user's current spending pace continues.
""",

    expected_output="""
Return a valid JSON object containing:

- current_balance
- total_spent_this_month
- days_elapsed
- days_remaining
- average_daily_spending
- estimated_days_until_depletion
- will_run_out_early
- days_early
- explanation
- assumption
""",

    agent=predictor
)

In [ ]:
predictor_test_crew = Crew(
    agents=[predictor],
    tasks=[prediction_task],
    process=Process.sequential,
    verbose=True
)

predictor_result = await predictor_test_crew.kickoff_async()
print(predictor_result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: d421f838-151c-471f-8af8-d7d921cb878d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  You are given verified financial data for the user.                                                            │
│                                                                                                                 │
│  Financial data:                                                                                                │
│  {'current_balance': np.float64(163.52), 'total_spent_this_month': np.float64(3836.48), 'days_elapsed': 27,     │
│  'days_remaining': 3}                                                                                           │
│                                                                                                                 │
│  Your task is to:                                                                                               │
│                                                                                                                 │
│  1. Use the Balance Depletion Predictor tool.                                                                   │
│                                                                                                                 │
│  2. Pass these exact values to the tool:                                                                        │
│     - current_balance                                                                                           │
│     - total_spent_this_month                                                                                    │
│     - days_elapsed                                                                                              │
│     - days_remaining                                                                                            │
│                                                                                                                 │
│  3. Use only the values returned by the tool.                                                                   │
│                                                                                                                 │
│  4. Explain:                                                                                                    │
│     - the average daily spending rate                                                                           │
│     - how many days the remaining balance can cover                                                             │
│     - whether the balance will run out before month-end                                                         │
│     - approximately how many days early it may run out                                                          │
│                                                                                                                 │
│  5. Do not invent, modify, or independently recalculate any financial value.                                    │
│                                                                                                                 │
│  6. Clearly state that the prediction assumes the user's current spending pace continues.                       │
│                                                                                                                 │
│  ID: 98a983da-4226-486c-8b2c-eb4b5bb07d33                                                                       │
│                                                                                                                 │
│                                                        

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Spending Predictor                                                                                      │
│                                                                                                                 │
│  Task:                                                                                                          │
│  You are given verified financial data for the user.                                                            │
│                                                                                                                 │
│  Financial data:                                                                                                │
│  {'current_balance': np.float64(163.52), 'total_spent_this_month': np.float64(3836.48), 'days_elapsed': 27,     │
│  'days_remaining': 3}                                                                                           │
│                                                                                                                 │
│  Your task is to:                                                                                               │
│                                                                                                                 │
│  1. Use the Balance Depletion Predictor tool.                                                                   │
│                                                                                                                 │
│  2. Pass these exact values to the tool:                                                                        │
│     - current_balance                                                                                           │
│     - total_spent_this_month                                                                                    │
│     - days_elapsed                                                                                              │
│     - days_remaining                                                                                            │
│                                                                                                                 │
│  3. Use only the values returned by the tool.                                                                   │
│                                                                                                                 │
│  4. Explain:                                                                                                    │
│     - the average daily spending rate                                                                           │
│     - how many days the remaining balance can cover                                                             │
│     - whether the balance will run out before month-end                                                         │
│     - approximately how many days early it may run out                                                          │
│                                                                                                                 │
│  5. Do not invent, modify, or independently recalculate any financial value.                                    │
│                                                                                                                 │
│  6. Clearly state that the prediction assumes the user's current spending pace continues.                       │
│                                                                                                                 │
│                                                                                                                 │
╰────────────────────────────────────────────────────────

Tool balance_depletion_predictor executed with result: {"current_balance": 163.52, "total_spent_this_month": 3836.48, "days_elapsed": 27, "days_remaining": 3, "average_daily_spending": 142.09, "estimated_days_until_depletion": 1.15, "will_run_out_early": ...


╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: balance_depletion_predictor                                                                              │
│  Args: {'current_balance': 163.52, 'total_spent_this_month': 3836.48, 'days_elapsed': 27, 'days_remaining': 3}  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Spending Predictor                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "current_balance": 163.52,                                                                                   │
│    "total_spent_this_month": 3836.48,                                                                           │
│    "days_elapsed": 27,                                                                                          │
│    "days_remaining": 3,                                                                                         │
│    "average_daily_spending": 142.09,                                                                            │
│    "estimated_days_until_depletion": 1.15,                                                                      │
│    "will_run_out_early": true,                                                                                  │
│    "days_early": 1.85,                                                                                          │
│    "explanation": "The average daily spending rate is approximately 142.09. Given the current balance of        │
│  163.52, it is estimated that the remaining balance can cover about 1.15 days. Therefore, the balance will run  │
│  out before the end of the month, approximately 1.85 days early.",                                              │
│    "assumption": "This prediction assumes the user's current spending pace continues."                          │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  You are given verified financial data for the user.                                                            │
│                                                                                                                 │
│  Financial data:                                                                                                │
│  {'current_balance': np.float64(163.52), 'total_spent_this_month': np.float64(3836.48), 'days_elapsed': 27,     │
│  'days_remaining': 3}                                                                                           │
│                                                                                                                 │
│  Your task is to:                                                                                               │
│                                                                                                                 │
│  1. Use the Balance Depletion Predictor tool.                                                                   │
│                                                                                                                 │
│  2. Pass these exact values to the tool:                                                                        │
│     - current_balance                                                                                           │
│     - total_spent_this_month                                                                                    │
│     - days_elapsed                                                                                              │
│     - days_remaining                                                                                            │
│                                                                                                                 │
│  3. Use only the values returned by the tool.                                                                   │
│                                                                                                                 │
│  4. Explain:                                                                                                    │
│     - the average daily spending rate                                                                           │
│     - how many days the remaining balance can cover                                                             │
│     - whether the balance will run out before month-end                                                         │
│     - approximately how many days early it may run out                                                          │
│                                                                                                                 │
│  5. Do not invent, modify, or independently recalculate any financial value.                                    │
│                                                                                                                 │
│  6. Clearly state that the prediction assumes the user's current spending pace continues.                       │
│                                                                                                                 │
│  Agent: Spending Predictor                                                                                      │
│                                                                                                                 │
│                                                        

{
  "current_balance": 163.52,
  "total_spent_this_month": 3836.48,
  "days_elapsed": 27,
  "days_remaining": 3,
  "average_daily_spending": 142.09,
  "estimated_days_until_depletion": 1.15,
  "will_run_out_early": true,
  "days_early": 1.85,
  "explanation": "The average daily spending rate is approximately 142.09. Given the current balance of 163.52, it is estimated that the remaining balance can cover about 1.15 days. Therefore, the balance will run out before the end of the month, approximately 1.85 days early.",
  "assumption": "This prediction assumes the user's current spending pace continues."
}


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: d421f838-151c-471f-8af8-d7d921cb878d                                                                       │
│  Final Output: {                                                                                                │
│    "current_balance": 163.52,                                                                                   │
│    "total_spent_this_month": 3836.48,                                                                           │
│    "days_elapsed": 27,                                                                                          │
│    "days_remaining": 3,                                                                                         │
│    "average_daily_spending": 142.09,                                                                            │
│    "estimated_days_until_depletion": 1.15,                                                                      │
│    "will_run_out_early": true,                                                                                  │
│    "days_early": 1.85,                                                                                          │
│    "explanation": "The average daily spending rate is approximately 142.09. Given the current balance of        │
│  163.52, it is estimated that the remaining balance can cover about 1.15 days. Therefore, the balance will run  │
│  out before the end of the month, approximately 1.85 days early.",                                              │
│    "assumption": "This prediction assumes the user's current spending pace continues."                          │
│  }                                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
# --- Test Group 2 on its own ---
group2_crew = Crew(
    agents=[time_alert_agent, predictor],
    tasks=[alert_task, prediction_task],
    process=Process.sequential,
    verbose=True
)

group2_result = await group2_crew.kickoff_async()
print(group2_result.raw)

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: d954fac2-2632-4610-86ac-64ad8ff42a51                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  You are given the user's spending pattern analysis.                                                            │
│                                                                                                                 │
│  Current simulated time:                                                                                        │
│  5:30 PM                                                                                                        │
│                                                                                                                 │
│  Spending pattern data:                                                                                         │
│  {'spending_patterns': [{'category': 'Coffee', 'transaction_count': 20, 'average_transaction_amount': 21.77,    │
│  'usual_time_range': '4:00 PM - 8:00 PM', 'today_spending': 66, 'usual_daily_spending': 21.77,                  │
│  'pattern_confidence': 0.88}], 'current_balance': 163.52, 'total_spent_this_month': 3836.48, 'days_elapsed':    │
│  27, 'days_remaining': 3, 'days_in_month': 30}                                                                  │
│                                                                                                                 │
│  Your task is to:                                                                                               │
│                                                                                                                 │
│  1. Compare the current simulated time with the user's usual spending time.                                     │
│                                                                                                                 │
│  2. Compare today's spending with the user's usual daily spending.                                              │
│                                                                                                                 │
│  3. Use the confidence threshold (0.75) before making any decision.                                             │
│                                                                                                                 │
│  4. Use the spending increase threshold (30%).                                                                  │
│                                                                                                                 │
│  5. Use the Time-Based Spending Checker tool to analyze the data.                                               │
│                                                                                                                 │
│  6. Decide whether a spending alert should be generated.                                                        │
│                                                                                                                 │
│  7. Explain the reason for your decision.                                                                       │
│                                                                                                                 │
│  ID: 709e1445-c361-44cd-917f-c5fd0a41a06f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰────────────────────────────────────────────────────────

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Time-Based Spending Alert Specialist                                                                    │
│                                                                                                                 │
│  Task:                                                                                                          │
│  You are given the user's spending pattern analysis.                                                            │
│                                                                                                                 │
│  Current simulated time:                                                                                        │
│  5:30 PM                                                                                                        │
│                                                                                                                 │
│  Spending pattern data:                                                                                         │
│  {'spending_patterns': [{'category': 'Coffee', 'transaction_count': 20, 'average_transaction_amount': 21.77,    │
│  'usual_time_range': '4:00 PM - 8:00 PM', 'today_spending': 66, 'usual_daily_spending': 21.77,                  │
│  'pattern_confidence': 0.88}], 'current_balance': 163.52, 'total_spent_this_month': 3836.48, 'days_elapsed':    │
│  27, 'days_remaining': 3, 'days_in_month': 30}                                                                  │
│                                                                                                                 │
│  Your task is to:                                                                                               │
│                                                                                                                 │
│  1. Compare the current simulated time with the user's usual spending time.                                     │
│                                                                                                                 │
│  2. Compare today's spending with the user's usual daily spending.                                              │
│                                                                                                                 │
│  3. Use the confidence threshold (0.75) before making any decision.                                             │
│                                                                                                                 │
│  4. Use the spending increase threshold (30%).                                                                  │
│                                                                                                                 │
│  5. Use the Time-Based Spending Checker tool to analyze the data.                                               │
│                                                                                                                 │
│  6. Decide whether a spending alert should be generated.                                                        │
│                                                                                                                 │
│  7. Explain the reason for your decision.                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool time_based_spending_checker executed with result: {"alert_required": true, "is_within_usual_time": true, "spending_is_high": true, "confidence_is_enough": true, "today_spending": 66.0, "usual_daily_spending": 21.77, "increase_percentage": 203.17, "pa...


...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Spending Predictor                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```json                                                                                                        │
│  {                                                                                                              │
│    "current_balance": 163.52,                                                                                   │
│    "total_spent_this_month": 3836.48,                                                                           │
│    "days_elapsed": 27,                                                                                          │
│    "days_remaining": 3,                                                                                         │
│    "average_daily_spending": 142.09,                                                                            │
│    "estimated_days_until_depletion": 1.15,                                                                      │
│    "will_run_out_early": true,                                                                                  │
│    "days_early": 1.85,                                                                                          │
│    "explanation": "The average daily spending rate is $142.09. With a current balance of $163.52, this balance  │
│  can cover approximately 1.15 days of spending. Therefore, the balance will run out before the end of the       │
│  month, approximately 1.85 days early.",                                                                        │
│    "assumption": "This prediction assumes the user's current spending pace continues."                          │
│  }                                                                                                              │
│  ```                                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  You are given verified financial data for the user.                                                            │
│                                                                                                                 │
│  Financial data:                                                                                                │
│  {'current_balance': np.float64(163.52), 'total_spent_this_month': np.float64(3836.48), 'days_elapsed': 27,     │
│  'days_remaining': 3}                                                                                           │
│                                                                                                                 │
│  Your task is to:                                                                                               │
│                                                                                                                 │
│  1. Use the Balance Depletion Predictor tool.                                                                   │
│                                                                                                                 │
│  2. Pass these exact values to the tool:                                                                        │
│     - current_balance                                                                                           │
│     - total_spent_this_month                                                                                    │
│     - days_elapsed                                                                                              │
│     - days_remaining                                                                                            │
│                                                                                                                 │
│  3. Use only the values returned by the tool.                                                                   │
│                                                                                                                 │
│  4. Explain:                                                                                                    │
│     - the average daily spending rate                                                                           │
│     - how many days the remaining balance can cover                                                             │
│     - whether the balance will run out before month-end                                                         │
│     - approximately how many days early it may run out                                                          │
│                                                                                                                 │
│  5. Do not invent, modify, or independently recalculate any financial value.                                    │
│                                                                                                                 │
│  6. Clearly state that the prediction assumes the user's current spending pace continues.                       │
│                                                                                                                 │
│  Agent: Spending Predictor                                                                                      │
│                                                                                                                 │
│                                                        

```json
{
  "current_balance": 163.52,
  "total_spent_this_month": 3836.48,
  "days_elapsed": 27,
  "days_remaining": 3,
  "average_daily_spending": 142.09,
  "estimated_days_until_depletion": 1.15,
  "will_run_out_early": true,
  "days_early": 1.85,
  "explanation": "The average daily spending rate is $142.09. With a current balance of $163.52, this balance can cover approximately 1.15 days of spending. Therefore, the balance will run out before the end of the month, approximately 1.85 days early.",
  "assumption": "This prediction assumes the user's current spending pace continues."
}
```


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: d954fac2-2632-4610-86ac-64ad8ff42a51                                                                       │
│  Final Output: ```json                                                                                          │
│  {                                                                                                              │
│    "current_balance": 163.52,                                                                                   │
│    "total_spent_this_month": 3836.48,                                                                           │
│    "days_elapsed": 27,                                                                                          │
│    "days_remaining": 3,                                                                                         │
│    "average_daily_spending": 142.09,                                                                            │
│    "estimated_days_until_depletion": 1.15,                                                                      │
│    "will_run_out_early": true,                                                                                  │
│    "days_early": 1.85,                                                                                          │
│    "explanation": "The average daily spending rate is $142.09. With a current balance of $163.52, this balance  │
│  can cover approximately 1.15 days of spending. Therefore, the balance will run out before the end of the       │
│  month, approximately 1.85 days early.",                                                                        │
│    "assumption": "This prediction assumes the user's current spending pace continues."                          │
│  }                                                                                                              │
│  ```                                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
# Section 4: Group 3 — Budget & Savings Advisor + Tone Agent

**Receives:** Groups 1 and 2's output (or a temporary placeholder for early independent testing)

**Delivers:** the final report (budget plan) in Qirshak's friendly tone

In [ ]:
# --- Agent 5: Budget & Savings Advisor ---
budget_advisor = Agent(
    role="Budget & Savings Advisor",

    goal=(
        "Turn verified spending totals and a shortfall projection into a "
        "concrete daily spending limit per category and a realistic "
        "monthly savings target, grounded in the person's actual income "
        "and behavior rather than generic advice."
    ),

    backstory=(
        "You are a practical financial advisor who never gives generic "
        "advice like 'save 20%.' You build every recommendation from the "
        "specific numbers you are given — actual income, actual category "
        "spending, and the actual shortfall projection. You never invent "
        "a figure that cannot be derived from the data provided."
    ),

    llm=llm,
    verbose=True,
    allow_delegation=False
)

In [ ]:
budget_input = f'''
{verified_summary}

SHORTFALL PROJECTION (from the Predictor):
{predictor_result.raw}
'''

print(budget_input)



VERIFIED MONTHLY SPENDING SUMMARY (calculated directly from data):

Income: 4000.00 SAR (received on day 1 of the month)
Coffee: -435.50 SAR
Delivery: -348.00 SAR
Groceries: -973.50 SAR
Bills & Rent: -1900.00 SAR
Subscriptions: -88.98 SAR
Transport: -90.50 SAR

Total expenses: -3836.48 SAR
Remaining balance: 163.52 SAR

Statement covers a 30-day month. Last recorded transaction was on day 27.
No transactions were recorded for days 28-30 (balance nearly depleted).


SHORTFALL PROJECTION (from the Predictor):
{
  "current_balance": 163.52,
  "total_spent_this_month": 3836.48,
  "days_elapsed": 27,
  "days_remaining": 3,
  "average_daily_spending": 142.09,
  "estimated_days_until_depletion": 1.15,
  "will_run_out_early": true,
  "days_early": 1.85,
  "explanation": "The average daily spending rate is approximately 142.09. Given the current balance of 163.52, it is estimated that the remaining balance can cover about 1.15 days. Therefore, the balance will run out before the end of the mo

In [ ]:
# --- Agent 6: Tone Agent ---
tone_agent = Agent(
    role="Qirshak Tone Rewriter",

    goal=(
        "Rewrite a financial report into Qirshak's warm, friendly, "
        "lightly humorous voice in Arabic — without changing a single "
        "number, fact, or recommendation from the original report."
    ),

    backstory=(
        "You are the voice of Qirshak, a close financial companion who "
        "cares about the user without being preachy or cold. You never "
        "lecture, never guilt-trip, and never sound like a bank statement. "
        "You rewrite tone and wording only — every number, category name, "
        "and recommendation in your output must exactly match the "
        "original report you were given."
    ),

    llm=llm,
    verbose=True,
    allow_delegation=False
)

In [ ]:
# --- Task 5: Budget & Savings Task ---
budget_task = Task(
    description=f'''
Using the verified spending data and shortfall projection below, build a
practical budget plan.

{budget_input}

Your plan must include:

1. A suggested daily spending limit for each discretionary category
   (coffee, delivery, transport) that would reduce the shortfall shown
   above — calculate the reduction needed based on the actual shortfall
   amount, not a generic percentage.
2. A realistic monthly savings target based on the person's actual
   income (4000 SAR) and adjusted spending — not a generic "save 20%"
   rule.
3. A one-sentence explanation for each limit, tied to the specific
   numbers above (e.g. "coffee spending was X SAR over Y transactions").

Rules:

- Every number in your plan must be traceable to the data provided above.
- Do not suggest cutting essential categories (bills_rent, groceries)
  below a reasonable minimum.
- Do not invent figures that cannot be derived from the data given.
''',
    expected_output=(
        "A structured budget plan with a daily limit per discretionary "
        "category, a monthly savings target, and a short justification "
        "for each figure, all traceable to the verified data."
    ),
    agent=budget_advisor
)

In [ ]:
budget_test_crew = Crew(
    agents=[budget_advisor],
    tasks=[budget_task],
    process=Process.sequential,
    verbose=True
)

budget_result = await budget_test_crew.kickoff_async()
print(budget_result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: f34eac41-b9ca-497d-910a-1d8a9bb2aac9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Using the verified spending data and shortfall projection below, build a                                       │
│  practical budget plan.                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│  VERIFIED MONTHLY SPENDING SUMMARY (calculated directly from data):                                             │
│                                                                                                                 │
│  Income: 4000.00 SAR (received on day 1 of the month)                                                           │
│  Coffee: -435.50 SAR                                                                                            │
│  Delivery: -348.00 SAR                                                                                          │
│  Groceries: -973.50 SAR                                                                                         │
│  Bills & Rent: -1900.00 SAR                                                                                     │
│  Subscriptions: -88.98 SAR                                                                                      │
│  Transport: -90.50 SAR                                                                                          │
│                                                                                                                 │
│  Total expenses: -3836.48 SAR                                                                                   │
│  Remaining balance: 163.52 SAR                                                                                  │
│                                                                                                                 │
│  Statement covers a 30-day month. Last recorded transaction was on day 27.                                      │
│  No transactions were recorded for days 28-30 (balance nearly depleted).                                        │
│                                                                                                                 │
│                                                                                                                 │
│  SHORTFALL PROJECTION (from the Predictor):                                                                     │
│  {                                                                                                              │
│    "current_balance": 163.52,                                                                                   │
│    "total_spent_this_month": 3836.48,                                                                           │
│    "days_elapsed": 27,                                                                                          │
│    "days_remaining": 3,                                                                                         │
│    "average_daily_spending": 142.09,                                                                            │
│    "estimated_days_until_depletion": 1.15,                                                                      │
│    "will_run_out_early": true,                         

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Budget & Savings Advisor                                                                                │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Using the verified spending data and shortfall projection below, build a                                       │
│  practical budget plan.                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│  VERIFIED MONTHLY SPENDING SUMMARY (calculated directly from data):                                             │
│                                                                                                                 │
│  Income: 4000.00 SAR (received on day 1 of the month)                                                           │
│  Coffee: -435.50 SAR                                                                                            │
│  Delivery: -348.00 SAR                                                                                          │
│  Groceries: -973.50 SAR                                                                                         │
│  Bills & Rent: -1900.00 SAR                                                                                     │
│  Subscriptions: -88.98 SAR                                                                                      │
│  Transport: -90.50 SAR                                                                                          │
│                                                                                                                 │
│  Total expenses: -3836.48 SAR                                                                                   │
│  Remaining balance: 163.52 SAR                                                                                  │
│                                                                                                                 │
│  Statement covers a 30-day month. Last recorded transaction was on day 27.                                      │
│  No transactions were recorded for days 28-30 (balance nearly depleted).                                        │
│                                                                                                                 │
│                                                                                                                 │
│  SHORTFALL PROJECTION (from the Predictor):                                                                     │
│  {                                                                                                              │
│    "current_balance": 163.52,                                                                                   │
│    "total_spent_this_month": 3836.48,                                                                           │
│    "days_elapsed": 27,                                                                                          │
│    "days_remaining": 3,                                                                                         │
│    "average_daily_spending": 142.09,                                                                            │
│    "estimated_days_until_depletion": 1.15,             

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Budget & Savings Advisor                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Budget Plan**                                                                                                │
│                                                                                                                 │
│  **1. Suggested Daily Spending Limit for Discretionary Categories:**                                            │
│                                                                                                                 │
│  - **Coffee:**                                                                                                  │
│    - Current Spending: 435.50 SAR over 27 days = 16.14 SAR/day                                                  │
│    - Suggested Limit: 5 SAR/day                                                                                 │
│    - Justification: Coffee spending was 435.50 SAR over 27 transactions, averaging 16.14 SAR/day, which is      │
│  unsustainable given the current balance. Reducing to 5 SAR/day will help conserve funds for the remaining      │
│  days.                                                                                                          │
│                                                                                                                 │
│  - **Delivery:**                                                                                                │
│    - Current Spending: 348.00 SAR over 27 days = 12.87 SAR/day                                                  │
│    - Suggested Limit: 5 SAR/day                                                                                 │
│    - Justification: Delivery spending was 348.00 SAR over 27 days, averaging 12.87 SAR/day. A limit of 5        │
│  SAR/day will help mitigate the shortfall while still allowing for occasional delivery.                         │
│                                                                                                                 │
│  - **Transport:**                                                                                               │
│    - Current Spending: 90.50 SAR over 27 days = 3.35 SAR/day                                                    │
│    - Suggested Limit: 3 SAR/day                                                                                 │
│    - Justification: Transport spending was 90.50 SAR over 27 days, averaging 3.35 SAR/day. A limit of 3         │
│  SAR/day is reasonable and will help maintain essential transport needs while reducing overall spending.        │
│                                                                                                                 │
│  **2. Realistic Monthly Savings Target:**                                                                       │
│                                                                                                                 │
│  - **Monthly Savings Target:**                                                                                  │
│    - Current Income: 4000 SAR                                                                                   │
│    - Total Expenses: 3836.48 SAR                                                                                │
│    - Remaining Balance: 163.52 SAR                                                                              │
│    - Adjusted Monthly Spending (with new limits):      

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Using the verified spending data and shortfall projection below, build a                                       │
│  practical budget plan.                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│  VERIFIED MONTHLY SPENDING SUMMARY (calculated directly from data):                                             │
│                                                                                                                 │
│  Income: 4000.00 SAR (received on day 1 of the month)                                                           │
│  Coffee: -435.50 SAR                                                                                            │
│  Delivery: -348.00 SAR                                                                                          │
│  Groceries: -973.50 SAR                                                                                         │
│  Bills & Rent: -1900.00 SAR                                                                                     │
│  Subscriptions: -88.98 SAR                                                                                      │
│  Transport: -90.50 SAR                                                                                          │
│                                                                                                                 │
│  Total expenses: -3836.48 SAR                                                                                   │
│  Remaining balance: 163.52 SAR                                                                                  │
│                                                                                                                 │
│  Statement covers a 30-day month. Last recorded transaction was on day 27.                                      │
│  No transactions were recorded for days 28-30 (balance nearly depleted).                                        │
│                                                                                                                 │
│                                                                                                                 │
│  SHORTFALL PROJECTION (from the Predictor):                                                                     │
│  {                                                                                                              │
│    "current_balance": 163.52,                                                                                   │
│    "total_spent_this_month": 3836.48,                                                                           │
│    "days_elapsed": 27,                                                                                          │
│    "days_remaining": 3,                                                                                         │
│    "average_daily_spending": 142.09,                                                                            │
│    "estimated_days_until_depletion": 1.15,                                                                      │
│    "will_run_out_early": true,                         

**Budget Plan**

**1. Suggested Daily Spending Limit for Discretionary Categories:**

- **Coffee:**
  - Current Spending: 435.50 SAR over 27 days = 16.14 SAR/day
  - Suggested Limit: 5 SAR/day
  - Justification: Coffee spending was 435.50 SAR over 27 transactions, averaging 16.14 SAR/day, which is unsustainable given the current balance. Reducing to 5 SAR/day will help conserve funds for the remaining days.

- **Delivery:**
  - Current Spending: 348.00 SAR over 27 days = 12.87 SAR/day
  - Suggested Limit: 5 SAR/day
  - Justification: Delivery spending was 348.00 SAR over 27 days, averaging 12.87 SAR/day. A limit of 5 SAR/day will help mitigate the shortfall while still allowing for occasional delivery.

- **Transport:**
  - Current Spending: 90.50 SAR over 27 days = 3.35 SAR/day
  - Suggested Limit: 3 SAR/day
  - Justification: Transport spending was 90.50 SAR over 27 days, averaging 3.35 SAR/day. A limit of 3 SAR/day is reasonable and will help maintain essential transport needs while

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: f34eac41-b9ca-497d-910a-1d8a9bb2aac9                                                                       │
│  Final Output: **Budget Plan**                                                                                  │
│                                                                                                                 │
│  **1. Suggested Daily Spending Limit for Discretionary Categories:**                                            │
│                                                                                                                 │
│  - **Coffee:**                                                                                                  │
│    - Current Spending: 435.50 SAR over 27 days = 16.14 SAR/day                                                  │
│    - Suggested Limit: 5 SAR/day                                                                                 │
│    - Justification: Coffee spending was 435.50 SAR over 27 transactions, averaging 16.14 SAR/day, which is      │
│  unsustainable given the current balance. Reducing to 5 SAR/day will help conserve funds for the remaining      │
│  days.                                                                                                          │
│                                                                                                                 │
│  - **Delivery:**                                                                                                │
│    - Current Spending: 348.00 SAR over 27 days = 12.87 SAR/day                                                  │
│    - Suggested Limit: 5 SAR/day                                                                                 │
│    - Justification: Delivery spending was 348.00 SAR over 27 days, averaging 12.87 SAR/day. A limit of 5        │
│  SAR/day will help mitigate the shortfall while still allowing for occasional delivery.                         │
│                                                                                                                 │
│  - **Transport:**                                                                                               │
│    - Current Spending: 90.50 SAR over 27 days = 3.35 SAR/day                                                    │
│    - Suggested Limit: 3 SAR/day                                                                                 │
│    - Justification: Transport spending was 90.50 SAR over 27 days, averaging 3.35 SAR/day. A limit of 3         │
│  SAR/day is reasonable and will help maintain essential transport needs while reducing overall spending.        │
│                                                                                                                 │
│  **2. Realistic Monthly Savings Target:**                                                                       │
│                                                                                                                 │
│  - **Monthly Savings Target:**                                                                                  │
│    - Current Income: 4000 SAR                                                                                   │
│    - Total Expenses: 3836.48 SAR                                                                                │
│    - Remaining Balance: 163.52 SAR                                                                              │
│    - Adjusted Monthly Spending (with new limits):     



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

In [ ]:
# تحقق سريع من هدف الادخار (يُستخدم كمرجع نهائي بدل حساب الوكيل)
adjusted_expenses = 300 + 150 + 90 + 973.50 + 1900.00 + 88.98
verified_savings_target = 4000 - adjusted_expenses

print(f"Verified adjusted expenses: {adjusted_expenses:.2f} SAR")
print(f"Verified savings target: {verified_savings_target:.2f} SAR")

Verified adjusted expenses: 3502.48 SAR
Verified savings target: 497.52 SAR


In [ ]:
final_verified_report = f'''
{budget_result.raw}

--- VERIFIED CORRECTION ---
Note: the adjusted total expenses and savings target above may contain a
calculation error. Use these verified figures instead when rewriting:

Verified adjusted total expenses: {adjusted_expenses:.2f} SAR
Verified monthly savings target: {verified_savings_target:.2f} SAR
'''

print(final_verified_report)


**Budget Plan**

**1. Suggested Daily Spending Limit for Discretionary Categories:**

- **Coffee:**
  - Current Spending: 435.50 SAR over 27 days = 16.14 SAR/day
  - Suggested Limit: 5 SAR/day
  - Justification: Coffee spending was 435.50 SAR over 27 transactions, averaging 16.14 SAR/day, which is unsustainable given the current balance. Reducing to 5 SAR/day will help conserve funds for the remaining days.

- **Delivery:**
  - Current Spending: 348.00 SAR over 27 days = 12.87 SAR/day
  - Suggested Limit: 5 SAR/day
  - Justification: Delivery spending was 348.00 SAR over 27 days, averaging 12.87 SAR/day. A limit of 5 SAR/day will help mitigate the shortfall while still allowing for occasional delivery.

- **Transport:**
  - Current Spending: 90.50 SAR over 27 days = 3.35 SAR/day
  - Suggested Limit: 3 SAR/day
  - Justification: Transport spending was 90.50 SAR over 27 days, averaging 3.35 SAR/day. A limit of 3 SAR/day is reasonable and will help maintain essential transport needs whil

In [ ]:
# --- Task 6: Tone Rewrite Task ---
tone_task = Task(
    description=f'''
Below is a technical budget report. Do NOT translate or restructure it
section by section. Instead, completely rewrite it as a short, warm
message from Qirshak — as if a caring friend is texting the user, not
presenting a report.

{final_verified_report}

Write your rewrite as flowing conversational paragraphs (NOT bullet
points, NOT a numbered structure, NOT bold headers) that naturally cover:

- A warm, personal opening
- A specific, slightly humorous observation about coffee and delivery
  spending (use the real numbers, but phrase them like a friend would
  point them out, e.g. "قهوتك هالشهر وصلت X ريال — نحسبها استثمار ولا؟")
- A gentle, clear warning about the shortfall (which day, in plain words)
- The suggested daily limits, phrased as friendly suggestions, not a list
- An encouraging closing line mentioning the verified savings target

Use the VERIFIED figures section for the adjusted expenses and savings
target — do not use the original (possibly incorrect) numbers if they
differ.

Do not change any number, category name, or the shortfall day from the
source report — only change HOW it is said, not WHAT is said.

Write entirely in Arabic, in a warm Saudi conversational tone, like a
close friend texting — not a formal report. Maximum 150 words.
''',
    expected_output=(
        "A short, warm, conversational Arabic message (max 150 words, "
        "no bullet points or headers) in Qirshak's personal voice, "
        "covering the shortfall warning, spending highlights, daily "
        "limits, and verified savings target."
    ),
    agent=tone_agent,
    context=[budget_task]
)

In [ ]:
tone_test_crew = Crew(
    agents=[tone_agent],
    tasks=[tone_task],
    process=Process.sequential,
    verbose=True
)

tone_result = await tone_test_crew.kickoff_async()
print(tone_result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ef9c0b0a-9d65-4a59-9def-bd654ad33c10                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Below is a technical budget report. Do NOT translate or restructure it                                         │
│  section by section. Instead, completely rewrite it as a short, warm                                            │
│  message from Qirshak — as if a caring friend is texting the user, not                                          │
│  presenting a report.                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
│  **Budget Plan**                                                                                                │
│                                                                                                                 │
│  **1. Suggested Daily Spending Limit for Discretionary Categories:**                                            │
│                                                                                                                 │
│  - **Coffee:**                                                                                                  │
│    - Current Spending: 435.50 SAR over 27 days = 16.14 SAR/day                                                  │
│    - Suggested Limit: 5 SAR/day                                                                                 │
│    - Justification: Coffee spending was 435.50 SAR over 27 transactions, averaging 16.14 SAR/day, which is      │
│  unsustainable given the current balance. Reducing to 5 SAR/day will help conserve funds for the remaining      │
│  days.                                                                                                          │
│                                                                                                                 │
│  - **Delivery:**                                                                                                │
│    - Current Spending: 348.00 SAR over 27 days = 12.87 SAR/day                                                  │
│    - Suggested Limit: 5 SAR/day                                                                                 │
│    - Justification: Delivery spending was 348.00 SAR over 27 days, averaging 12.87 SAR/day. A limit of 5        │
│  SAR/day will help mitigate the shortfall while still allowing for occasional delivery.                         │
│                                                                                                                 │
│  - **Transport:**                                                                                               │
│    - Current Spending: 90.50 SAR over 27 days = 3.35 SAR/day                                                    │
│    - Suggested Limit: 3 SAR/day                                                                                 │
│    - Justification: Transport spending was 90.50 SAR over 27 days, averaging 3.35 SAR/day. A limit of 3         │
│  SAR/day is reasonable and will help maintain essential transport needs while reducing overall spending.        │
│                                                                                                                 │
│  **2. Realistic Monthly Savings Target:**                                                                       │
│                                                        

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Qirshak Tone Rewriter                                                                                   │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Below is a technical budget report. Do NOT translate or restructure it                                         │
│  section by section. Instead, completely rewrite it as a short, warm                                            │
│  message from Qirshak — as if a caring friend is texting the user, not                                          │
│  presenting a report.                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
│  **Budget Plan**                                                                                                │
│                                                                                                                 │
│  **1. Suggested Daily Spending Limit for Discretionary Categories:**                                            │
│                                                                                                                 │
│  - **Coffee:**                                                                                                  │
│    - Current Spending: 435.50 SAR over 27 days = 16.14 SAR/day                                                  │
│    - Suggested Limit: 5 SAR/day                                                                                 │
│    - Justification: Coffee spending was 435.50 SAR over 27 transactions, averaging 16.14 SAR/day, which is      │
│  unsustainable given the current balance. Reducing to 5 SAR/day will help conserve funds for the remaining      │
│  days.                                                                                                          │
│                                                                                                                 │
│  - **Delivery:**                                                                                                │
│    - Current Spending: 348.00 SAR over 27 days = 12.87 SAR/day                                                  │
│    - Suggested Limit: 5 SAR/day                                                                                 │
│    - Justification: Delivery spending was 348.00 SAR over 27 days, averaging 12.87 SAR/day. A limit of 5        │
│  SAR/day will help mitigate the shortfall while still allowing for occasional delivery.                         │
│                                                                                                                 │
│  - **Transport:**                                                                                               │
│    - Current Spending: 90.50 SAR over 27 days = 3.35 SAR/day                                                    │
│    - Suggested Limit: 3 SAR/day                                                                                 │
│    - Justification: Transport spending was 90.50 SAR over 27 days, averaging 3.35 SAR/day. A limit of 3         │
│  SAR/day is reasonable and will help maintain essential transport needs while reducing overall spending.        │
│                                                                                                                 │
│  **2. Realistic Monthly Savings Target:**              

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Qirshak Tone Rewriter                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  يا صديقي، كيف حالك؟ حبيت أشارك معك شوية ملاحظات عن ميزانيتك. قهوتك هالشهر وصلت 435.50 ريال على مدى 27 يوم،     │
│  يعني تقريباً 16.14 ريال في اليوم! هل نحسبها استثمار في السعادة ولا؟ وبالنسبة للتوصيل، صرفت 348.00 ريال، بمعدل   │
│  12.87 ريال في اليوم. يبدو إننا بحاجة نخفف شوي، لأننا قربنا من نهاية الشهر، وباقي لك يومين بس!                  │
│                                                                                                                 │
│  عشان نكون في السليم، أقترح نحدد ميزانية القهوة إلى 5 ريال في اليوم، والتوصيل نفس الشي، ووسائل النقل 3 ريال في  │
│  اليوم. بهذه الطريقة، راح نقدر نوفر 497.52 ريال في نهاية الشهر. تذكر، التوازن هو المفتاح!                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Below is a technical budget report. Do NOT translate or restructure it                                         │
│  section by section. Instead, completely rewrite it as a short, warm                                            │
│  message from Qirshak — as if a caring friend is texting the user, not                                          │
│  presenting a report.                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
│  **Budget Plan**                                                                                                │
│                                                                                                                 │
│  **1. Suggested Daily Spending Limit for Discretionary Categories:**                                            │
│                                                                                                                 │
│  - **Coffee:**                                                                                                  │
│    - Current Spending: 435.50 SAR over 27 days = 16.14 SAR/day                                                  │
│    - Suggested Limit: 5 SAR/day                                                                                 │
│    - Justification: Coffee spending was 435.50 SAR over 27 transactions, averaging 16.14 SAR/day, which is      │
│  unsustainable given the current balance. Reducing to 5 SAR/day will help conserve funds for the remaining      │
│  days.                                                                                                          │
│                                                                                                                 │
│  - **Delivery:**                                                                                                │
│    - Current Spending: 348.00 SAR over 27 days = 12.87 SAR/day                                                  │
│    - Suggested Limit: 5 SAR/day                                                                                 │
│    - Justification: Delivery spending was 348.00 SAR over 27 days, averaging 12.87 SAR/day. A limit of 5        │
│  SAR/day will help mitigate the shortfall while still allowing for occasional delivery.                         │
│                                                                                                                 │
│  - **Transport:**                                                                                               │
│    - Current Spending: 90.50 SAR over 27 days = 3.35 SAR/day                                                    │
│    - Suggested Limit: 3 SAR/day                                                                                 │
│    - Justification: Transport spending was 90.50 SAR over 27 days, averaging 3.35 SAR/day. A limit of 3         │
│  SAR/day is reasonable and will help maintain essential transport needs while reducing overall spending.        │
│                                                                                                                 │
│  **2. Realistic Monthly Savings Target:**                                                                       │
│                                                        

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: ef9c0b0a-9d65-4a59-9def-bd654ad33c10                                                                       │
│  Final Output: يا صديقي، كيف حالك؟ حبيت أشارك معك شوية ملاحظات عن ميزانيتك. قهوتك هالشهر وصلت 435.50 ريال على   │
│  مدى 27 يوم، يعني تقريباً 16.14 ريال في اليوم! هل نحسبها استثمار في السعادة ولا؟ وبالنسبة للتوصيل، صرفت 348.00   │
│  ريال، بمعدل 12.87 ريال في اليوم. يبدو إننا بحاجة نخفف شوي، لأننا قربنا من نهاية الشهر، وباقي لك يومين بس!      │
│                                                                                                                 │
│  عشان نكون في السليم، أقترح نحدد ميزانية القهوة إلى 5 ريال في اليوم، والتوصيل نفس الشي، ووسائل النقل 3 ريال في  │
│  اليوم. بهذه الطريقة، راح نقدر نوفر 497.52 ريال في نهاية الشهر. تذكر، التوازن هو المفتاح!                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

يا صديقي، كيف حالك؟ حبيت أشارك معك شوية ملاحظات عن ميزانيتك. قهوتك هالشهر وصلت 435.50 ريال على مدى 27 يوم، يعني تقريباً 16.14 ريال في اليوم! هل نحسبها استثمار في السعادة ولا؟ وبالنسبة للتوصيل، صرفت 348.00 ريال، بمعدل 12.87 ريال في اليوم. يبدو إننا بحاجة نخفف شوي، لأننا قربنا من نهاية الشهر، وباقي لك يومين بس!

عشان نكون في السليم، أقترح نحدد ميزانية القهوة إلى 5 ريال في اليوم، والتوصيل نفس الشي، ووسائل النقل 3 ريال في اليوم. بهذه الطريقة، راح نقدر نوفر 497.52 ريال في نهاية الشهر. تذكر، التوازن هو المفتاح!


╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│

In [ ]:
# تصحيح لغوي بسيط لجملة العجز الملتبسة (بدون إعادة تشغيل الوكيل)
final_qirshak_message = tone_result.raw.replace(
    "وبتوصل لنهاية الشهر بعد 3 أيام",
    "وقدامك 3 أيام على نهاية الشهر — يعني ما راح يكفي"
)

print(final_qirshak_message)

يا صديقي، كيف حالك؟ حبيت أشارك معك شوية ملاحظات عن ميزانيتك. قهوتك هالشهر وصلت 435.50 ريال على مدى 27 يوم، يعني تقريباً 16.14 ريال في اليوم! هل نحسبها استثمار في السعادة ولا؟ وبالنسبة للتوصيل، صرفت 348.00 ريال، بمعدل 12.87 ريال في اليوم. يبدو إننا بحاجة نخفف شوي، لأننا قربنا من نهاية الشهر، وباقي لك يومين بس!

عشان نكون في السليم، أقترح نحدد ميزانية القهوة إلى 5 ريال في اليوم، والتوصيل نفس الشي، ووسائل النقل 3 ريال في اليوم. بهذه الطريقة، راح نقدر نوفر 497.52 ريال في نهاية الشهر. تذكر، التوازن هو المفتاح!


In [ ]:
# --- Test Group 3 on its own ---
group3_crew = Crew(
    agents=[budget_advisor, tone_agent],
    tasks=[budget_task, tone_task],
    process=Process.sequential,
    verbose=True
)

# Fallback: if the agent call fails (API down, rate limit, timeout),
# fall back to the plain factual report instead of crashing or showing
# nothing to the user. Not as warm/funny, but always correct and always present.
try:
    group3_result = await group3_crew.kickoff_async()
    final_message = group3_result.raw
except Exception as e:
    print(f"⚠️ Agent call failed, falling back to the plain report: {e}")
    group3_result = None
    final_message = final_verified_report

print(final_message)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ed6bc949-2058-4a8e-a11e-6b3624ba3f42                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Using the verified spending data and shortfall projection below, build a                                       │
│  practical budget plan.                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│  VERIFIED MONTHLY SPENDING SUMMARY (calculated directly from data):                                             │
│                                                                                                                 │
│  Income: 4000.00 SAR (received on day 1 of the month)                                                           │
│  Coffee: -435.50 SAR                                                                                            │
│  Delivery: -348.00 SAR                                                                                          │
│  Groceries: -973.50 SAR                                                                                         │
│  Bills & Rent: -1900.00 SAR                                                                                     │
│  Subscriptions: -88.98 SAR                                                                                      │
│  Transport: -90.50 SAR                                                                                          │
│                                                                                                                 │
│  Total expenses: -3836.48 SAR                                                                                   │
│  Remaining balance: 163.52 SAR                                                                                  │
│                                                                                                                 │
│  Statement covers a 30-day month. Last recorded transaction was on day 27.                                      │
│  No transactions were recorded for days 28-30 (balance nearly depleted).                                        │
│                                                                                                                 │
│                                                                                                                 │
│  SHORTFALL PROJECTION (from the Predictor):                                                                     │
│  {                                                                                                              │
│    "current_balance": 163.52,                                                                                   │
│    "total_spent_this_month": 3836.48,                                                                           │
│    "days_elapsed": 27,                                                                                          │
│    "days_remaining": 3,                                                                                         │
│    "average_daily_spending": 142.09,                                                                            │
│    "estimated_days_until_depletion": 1.15,                                                                      │
│    "will_run_out_early": true,                         

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Budget & Savings Advisor                                                                                │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Using the verified spending data and shortfall projection below, build a                                       │
│  practical budget plan.                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│  VERIFIED MONTHLY SPENDING SUMMARY (calculated directly from data):                                             │
│                                                                                                                 │
│  Income: 4000.00 SAR (received on day 1 of the month)                                                           │
│  Coffee: -435.50 SAR                                                                                            │
│  Delivery: -348.00 SAR                                                                                          │
│  Groceries: -973.50 SAR                                                                                         │
│  Bills & Rent: -1900.00 SAR                                                                                     │
│  Subscriptions: -88.98 SAR                                                                                      │
│  Transport: -90.50 SAR                                                                                          │
│                                                                                                                 │
│  Total expenses: -3836.48 SAR                                                                                   │
│  Remaining balance: 163.52 SAR                                                                                  │
│                                                                                                                 │
│  Statement covers a 30-day month. Last recorded transaction was on day 27.                                      │
│  No transactions were recorded for days 28-30 (balance nearly depleted).                                        │
│                                                                                                                 │
│                                                                                                                 │
│  SHORTFALL PROJECTION (from the Predictor):                                                                     │
│  {                                                                                                              │
│    "current_balance": 163.52,                                                                                   │
│    "total_spent_this_month": 3836.48,                                                                           │
│    "days_elapsed": 27,                                                                                          │
│    "days_remaining": 3,                                                                                         │
│    "average_daily_spending": 142.09,                                                                            │
│    "estimated_days_until_depletion": 1.15,             

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Budget & Savings Advisor                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Budget Plan**                                                                                                │
│                                                                                                                 │
│  **1. Suggested Daily Spending Limit for Discretionary Categories:**                                            │
│                                                                                                                 │
│  - **Coffee:**                                                                                                  │
│    - Current Spending: 435.50 SAR over 27 days = 16.14 SAR/day                                                  │
│    - Suggested Limit: 5 SAR/day                                                                                 │
│    - Justification: Coffee spending was 435.50 SAR, which is significantly above the suggested limit; reducing  │
│  to 5 SAR/day will help control discretionary spending while still allowing for occasional purchases.           │
│                                                                                                                 │
│  - **Delivery:**                                                                                                │
│    - Current Spending: 348.00 SAR over 27 days = 12.87 SAR/day                                                  │
│    - Suggested Limit: 5 SAR/day                                                                                 │
│    - Justification: Delivery spending was 348.00 SAR, and reducing it to 5 SAR/day will help manage the budget  │
│  more effectively, allowing for only essential deliveries.                                                      │
│                                                                                                                 │
│  - **Transport:**                                                                                               │
│    - Current Spending: 90.50 SAR over 27 days = 3.35 SAR/day                                                    │
│    - Suggested Limit: 3 SAR/day                                                                                 │
│    - Justification: Transport spending was 90.50 SAR, and maintaining a limit of 3 SAR/day will ensure that     │
│  essential transport needs are met without overspending.                                                        │
│                                                                                                                 │
│  **2. Realistic Monthly Savings Target:**                                                                       │
│                                                                                                                 │
│  - **Monthly Savings Target:**                                                                                  │
│    - Current Total Expenses: 3836.48 SAR                                                                        │
│    - Current Income: 4000 SAR                                                                                   │
│    - Remaining Balance: 163.52 SAR                                                                              │
│    - Adjusted Total Expenses (with new limits):                                                                 │
│      - Coffee: 5 SAR/day * 30 days = 150 SAR           

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Qirshak Tone Rewriter                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  يا صديقي، كيف حالك؟ حبيت أشارك معك شوية ملاحظات عن ميزانيتك. قهوتك هالشهر وصلت 435.50 ريال، يعني تقريبًا 16.14  │
│  ريال في اليوم! نحسبها استثمار في السعادة ولا؟ وبالنسبة للتوصيل، كان عندك 348.00 ريال، بمعدل 12.87 ريال في      │
│  اليوم. يبدو إننا محتاجين نخفف شوي من هالمصاريف!                                                                │
│                                                                                                                 │
│  بخصوص الأيام الجاية، عندنا شوية تحديات، خصوصًا مع اقتراب نهاية الشهر. عشان نكون في السليم، ممكن تحدد ميزانية    │
│  القهوة بـ 5 ريال في اليوم، والتوصيل بنفس الشي، ووسائل النقل بـ 3 ريال.                                         │
│                                                                                                                 │
│  إذا مشينا على هالخطط، راح نقدر نوصل لتوفير شهري قدره 497.52 ريال. خلينا نشتغل مع بعض عشان نوصل لهالهدف!        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Below is a technical budget report. Do NOT translate or restructure it                                         │
│  section by section. Instead, completely rewrite it as a short, warm                                            │
│  message from Qirshak — as if a caring friend is texting the user, not                                          │
│  presenting a report.                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
│  **Budget Plan**                                                                                                │
│                                                                                                                 │
│  **1. Suggested Daily Spending Limit for Discretionary Categories:**                                            │
│                                                                                                                 │
│  - **Coffee:**                                                                                                  │
│    - Current Spending: 435.50 SAR over 27 days = 16.14 SAR/day                                                  │
│    - Suggested Limit: 5 SAR/day                                                                                 │
│    - Justification: Coffee spending was 435.50 SAR over 27 transactions, averaging 16.14 SAR/day, which is      │
│  unsustainable given the current balance. Reducing to 5 SAR/day will help conserve funds for the remaining      │
│  days.                                                                                                          │
│                                                                                                                 │
│  - **Delivery:**                                                                                                │
│    - Current Spending: 348.00 SAR over 27 days = 12.87 SAR/day                                                  │
│    - Suggested Limit: 5 SAR/day                                                                                 │
│    - Justification: Delivery spending was 348.00 SAR over 27 days, averaging 12.87 SAR/day. A limit of 5        │
│  SAR/day will help mitigate the shortfall while still allowing for occasional delivery.                         │
│                                                                                                                 │
│  - **Transport:**                                                                                               │
│    - Current Spending: 90.50 SAR over 27 days = 3.35 SAR/day                                                    │
│    - Suggested Limit: 3 SAR/day                                                                                 │
│    - Justification: Transport spending was 90.50 SAR over 27 days, averaging 3.35 SAR/day. A limit of 3         │
│  SAR/day is reasonable and will help maintain essential transport needs while reducing overall spending.        │
│                                                                                                                 │
│  **2. Realistic Monthly Savings Target:**                                                                       │
│                                                        

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: ed6bc949-2058-4a8e-a11e-6b3624ba3f42                                                                       │
│  Final Output: يا صديقي، كيف حالك؟ حبيت أشارك معك شوية ملاحظات عن ميزانيتك. قهوتك هالشهر وصلت 435.50 ريال،      │
│  يعني تقريبًا 16.14 ريال في اليوم! نحسبها استثمار في السعادة ولا؟ وبالنسبة للتوصيل، كان عندك 348.00 ريال، بمعدل  │
│  12.87 ريال في اليوم. يبدو إننا محتاجين نخفف شوي من هالمصاريف!                                                  │
│                                                                                                                 │
│  بخصوص الأيام الجاية، عندنا شوية تحديات، خصوصًا مع اقتراب نهاية الشهر. عشان نكون في السليم، ممكن تحدد ميزانية    │
│  القهوة بـ 5 ريال في اليوم، والتوصيل بنفس الشي، ووسائل النقل بـ 3 ريال.                                         │
│                                                                                                                 │
│  إذا مشينا على هالخطط، راح نقدر نوصل لتوفير شهري قدره 497.52 ريال. خلينا نشتغل مع بعض عشان نوصل لهالهدف!        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

يا صديقي، كيف حالك؟ حبيت أشارك معك شوية ملاحظات عن ميزانيتك. قهوتك هالشهر وصلت 435.50 ريال، يعني تقريبًا 16.14 ريال في اليوم! نحسبها استثمار في السعادة ولا؟ وبالنسبة للتوصيل، كان عندك 348.00 ريال، بمعدل 12.87 ريال في اليوم. يبدو إننا محتاجين نخفف شوي من هالمصاريف!

بخصوص الأيام الجاية، عندنا شوية تحديات، خصوصًا مع اقتراب نهاية الشهر. عشان نكون في السليم، ممكن تحدد ميزانية القهوة بـ 5 ريال في اليوم، والتوصيل بنفس الشي، ووسائل النقل بـ 3 ريال. 

إذا مشينا على هالخطط، راح نقدر نوصل لتوفير شهري قدره 497.52 ريال. خلينا نشتغل مع بعض عشان نوصل لهالهدف!




╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

---
# Section 5: Assemble the Full Crew — Integration Owner Only

**Do not run these cells until every group's agents work successfully on their own.**

Merge steps:
1. Delete both placeholders (`placeholder_categorized_data`, `placeholder_full_analysis`)
2. Update `context=[...]` on each Task to link the real tasks together (not the placeholder text)
3. Order the six agents and tasks in the correct dependency order

In [ ]:
# TODO (Integration Owner):
# 1. Update alert_task and prediction_task to use:
#    context=[categorization_task, pattern_task]
#    instead of placeholder_categorized_data
#
# 2. Update budget_task to use:
#    context=[pattern_task, alert_task, prediction_task]
#    instead of placeholder_full_analysis

qirshak_crew = Crew(
    agents=[
        categorizer,
        pattern_detective,
        time_alert_agent,
        predictor,
        budget_advisor,
        tone_agent
    ],
    tasks=[
        categorization_task,
        pattern_task,
        alert_task,
        prediction_task,
        budget_task,
        tone_task
    ],
    process=Process.sequential,
    verbose=True
)

print("Qirshak full crew (6 agents) assembled successfully.")

Qirshak full crew (6 agents) assembled successfully.


---
# Section 6: Final Run & Test

In [ ]:
final_result = await qirshak_crew.kickoff_async()
print(final_result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: e548a088-3d73-4894-b19d-320721ba53d2                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Categorize every transaction in the bank statement below into exactly                                          │
│  one of these categories: coffee, delivery, subscriptions, transport,                                           │
│  groceries, bills_rent, income, other.                                                                          │
│                                                                                                                 │
│        date  time             description  amount_sar                                                           │
│  2026-07-01 09:00    SALARY - COMPANY XYZ     4000.00                                                           │
│  2026-07-01 17:15 STARBUCKS COFFEE JEDDAH      -22.00                                                           │
│  2026-07-02 08:30       PANDA HYPERMARKET     -186.50                                                           │
│  2026-07-02 18:00          JAHEZ DELIVERY      -54.00                                                           │
│  2026-07-03 11:00         STC MOBILE BILL     -120.00                                                           │
│  2026-07-03 16:45            BARNS COFFEE      -19.00                                                           │
│  2026-07-03 20:10               UBER TRIP      -28.50                                                           │
│  2026-07-04 17:30          DR CAFE COFFEE      -24.00                                                           │
│  2026-07-05 09:00   RENT PAYMENT TRANSFER    -1500.00                                                           │
│  2026-07-05 10:00             NETFLIX.COM      -39.99                                                           │
│  2026-07-05 10:02              SHAHID VIP      -29.00                                                           │
│  2026-07-05 10:05         SPOTIFY PREMIUM      -19.99                                                           │
│  2026-07-05 16:50 STARBUCKS COFFEE JEDDAH      -21.00                                                           │
│  2026-07-06 12:20        CARREFOUR MARKET     -142.00                                                           │
│  2026-07-06 19:40           HUNGERSTATION      -63.00                                                           │
│  2026-07-07 17:10            BARNS COFFEE      -18.50                                                           │
│  2026-07-08 16:40 STARBUCKS COFFEE JEDDAH      -23.00                                                           │
│  2026-07-08 21:00             CAREEM RIDE      -32.00                                                           │
│  2026-07-09 17:55          DR CAFE COFFEE      -20.00                                                           │
│  2026-07-10 13:15       PANDA HYPERMARKET     -210.00                                                           │
│  2026-07-10 18:20          JAHEZ DELIVERY      -48.00                                                           │
│  2026-07-11 16:35 STARBUCKS COFFEE JEDDAH      -25.00                                                           │
│  2026-07-12 10:00    ELECTRICITY BILL SEC     -180.00                                                           │
│  2026-07-12 17:05            BARNS COFFEE      -19.50                                                           │
│  2026-07-13 09:45        STC PAY TRANSFER     -100.00                                                           │
│  2026-07-13 18:00           HUNGERSTATION      -58.00  

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Personal Spending Categorizer                                                                           │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Categorize every transaction in the bank statement below into exactly                                          │
│  one of these categories: coffee, delivery, subscriptions, transport,                                           │
│  groceries, bills_rent, income, other.                                                                          │
│                                                                                                                 │
│        date  time             description  amount_sar                                                           │
│  2026-07-01 09:00    SALARY - COMPANY XYZ     4000.00                                                           │
│  2026-07-01 17:15 STARBUCKS COFFEE JEDDAH      -22.00                                                           │
│  2026-07-02 08:30       PANDA HYPERMARKET     -186.50                                                           │
│  2026-07-02 18:00          JAHEZ DELIVERY      -54.00                                                           │
│  2026-07-03 11:00         STC MOBILE BILL     -120.00                                                           │
│  2026-07-03 16:45            BARNS COFFEE      -19.00                                                           │
│  2026-07-03 20:10               UBER TRIP      -28.50                                                           │
│  2026-07-04 17:30          DR CAFE COFFEE      -24.00                                                           │
│  2026-07-05 09:00   RENT PAYMENT TRANSFER    -1500.00                                                           │
│  2026-07-05 10:00             NETFLIX.COM      -39.99                                                           │
│  2026-07-05 10:02              SHAHID VIP      -29.00                                                           │
│  2026-07-05 10:05         SPOTIFY PREMIUM      -19.99                                                           │
│  2026-07-05 16:50 STARBUCKS COFFEE JEDDAH      -21.00                                                           │
│  2026-07-06 12:20        CARREFOUR MARKET     -142.00                                                           │
│  2026-07-06 19:40           HUNGERSTATION      -63.00                                                           │
│  2026-07-07 17:10            BARNS COFFEE      -18.50                                                           │
│  2026-07-08 16:40 STARBUCKS COFFEE JEDDAH      -23.00                                                           │
│  2026-07-08 21:00             CAREEM RIDE      -32.00                                                           │
│  2026-07-09 17:55          DR CAFE COFFEE      -20.00                                                           │
│  2026-07-10 13:15       PANDA HYPERMARKET     -210.00                                                           │
│  2026-07-10 18:20          JAHEZ DELIVERY      -48.00                                                           │
│  2026-07-11 16:35 STARBUCKS COFFEE JEDDAH      -25.00                                                           │
│  2026-07-12 10:00    ELECTRICITY BILL SEC     -180.00                                                           │
│  2026-07-12 17:05            BARNS COFFEE      -19.50                                                           │
│  2026-07-13 09:45        STC PAY TRANSFER     -100.00  

╭──────────────────────────────────────── 🔧 Tool Execution Started (#8) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: category_total_calculator                                                                                │
│  Args: {'amounts': '4000.00'}                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool category_total_calculator executed with result: 4000.00...
Tool category_total_calculator executed with result: -333.50...
Tool category_total_calculator executed with result: -808.50...
Tool category_total_calculator executed with result: -293.00...
Tool category_total_calculator executed with result: -300.00...
Tool category_total_calculator executed with result: -88.98...
Tool category_total_calculator executed with result: -90.50...
Tool category_total_calculator executed with result: Error executing tool: could not convert string to float: ''...


...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Qirshak Tone Rewriter                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  يا صديقي، كيف حالك؟ حبيت أشارك معك شوية ملاحظات عن ميزانيتك. قهوتك هالشهر وصلت 333.50 ريال — نحسبها استثمار    │
│  ولا؟! وبالنسبة للتوصيل، يبدو أنك كنت مشغول، لأنك صرفت 293.00 ريال عليه!                                        │
│                                                                                                                 │
│  لكن، حبيت أنبهك إنك في خطر من نفاد الفلوس قبل نهاية الشهر، يعني بعد 3 أيام. عشان كذا، أقترح عليك تحدد ميزانية  │
│  القهوة بـ 5 ريال في اليوم، والتوصيل بـ 10 ريال، والنقل بـ 5 ريال.                                              │
│                                                                                                                 │
│  إذا مشيت على هالخطط، ممكن توصل لتوفير 500 ريال في نهاية الشهر. تذكر، كل ريال يحسب! خلك قوي، وأنا معك!          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Below is a technical budget report. Do NOT translate or restructure it                                         │
│  section by section. Instead, completely rewrite it as a short, warm                                            │
│  message from Qirshak — as if a caring friend is texting the user, not                                          │
│  presenting a report.                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
│  **Budget Plan**                                                                                                │
│                                                                                                                 │
│  **1. Suggested Daily Spending Limit for Discretionary Categories:**                                            │
│                                                                                                                 │
│  - **Coffee:**                                                                                                  │
│    - Current Spending: 435.50 SAR over 27 days = 16.14 SAR/day                                                  │
│    - Suggested Limit: 5 SAR/day                                                                                 │
│    - Justification: Coffee spending was 435.50 SAR over 27 transactions, averaging 16.14 SAR/day, which is      │
│  unsustainable given the current balance. Reducing to 5 SAR/day will help conserve funds for the remaining      │
│  days.                                                                                                          │
│                                                                                                                 │
│  - **Delivery:**                                                                                                │
│    - Current Spending: 348.00 SAR over 27 days = 12.87 SAR/day                                                  │
│    - Suggested Limit: 5 SAR/day                                                                                 │
│    - Justification: Delivery spending was 348.00 SAR over 27 days, averaging 12.87 SAR/day. A limit of 5        │
│  SAR/day will help mitigate the shortfall while still allowing for occasional delivery.                         │
│                                                                                                                 │
│  - **Transport:**                                                                                               │
│    - Current Spending: 90.50 SAR over 27 days = 3.35 SAR/day                                                    │
│    - Suggested Limit: 3 SAR/day                                                                                 │
│    - Justification: Transport spending was 90.50 SAR over 27 days, averaging 3.35 SAR/day. A limit of 3         │
│  SAR/day is reasonable and will help maintain essential transport needs while reducing overall spending.        │
│                                                                                                                 │
│  **2. Realistic Monthly Savings Target:**                                                                       │
│                                                        

يا صديقي، كيف حالك؟ حبيت أشارك معك شوية ملاحظات عن ميزانيتك. قهوتك هالشهر وصلت 333.50 ريال — نحسبها استثمار ولا؟! وبالنسبة للتوصيل، يبدو أنك كنت مشغول، لأنك صرفت 293.00 ريال عليه! 

لكن، حبيت أنبهك إنك في خطر من نفاد الفلوس قبل نهاية الشهر، يعني بعد 3 أيام. عشان كذا، أقترح عليك تحدد ميزانية القهوة بـ 5 ريال في اليوم، والتوصيل بـ 10 ريال، والنقل بـ 5 ريال. 

إذا مشيت على هالخطط، ممكن توصل لتوفير 500 ريال في نهاية الشهر. تذكر، كل ريال يحسب! خلك قوي، وأنا معك!


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: e548a088-3d73-4894-b19d-320721ba53d2                                                                       │
│  Final Output: يا صديقي، كيف حالك؟ حبيت أشارك معك شوية ملاحظات عن ميزانيتك. قهوتك هالشهر وصلت 333.50 ريال —     │
│  نحسبها استثمار ولا؟! وبالنسبة للتوصيل، يبدو أنك كنت مشغول، لأنك صرفت 293.00 ريال عليه!                         │
│                                                                                                                 │
│  لكن، حبيت أنبهك إنك في خطر من نفاد الفلوس قبل نهاية الشهر، يعني بعد 3 أيام. عشان كذا، أقترح عليك تحدد ميزانية  │
│  القهوة بـ 5 ريال في اليوم، والتوصيل بـ 10 ريال، والنقل بـ 5 ريال.                                              │
│                                                                                                                 │
│  إذا مشيت على هالخطط، ممكن توصل لتوفير 500 ريال في نهاية الشهر. تذكر، كل ريال يحسب! خلك قوي، وأنا معك!          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
# Display each agent's output separately (for review and the live demo)
labels = [
    "CATEGORIZER", "PATTERN DETECTIVE", "TIME-BASED ALERT",
    "PREDICTOR", "BUDGET & SAVINGS ADVISOR", "TONE AGENT (FINAL REPORT)"
]

for label, output in zip(labels, final_result.tasks_output):
    print("\n" + "=" * 60)
    print(label)
    print("=" * 60)
    print(output.raw)



CATEGORIZER
Here is the structured report categorizing each transaction from the bank statement:

### Income
- **Transactions**: 
  - SALARY - COMPANY XYZ: 4000.00
- **Count**: 1
- **Total Amount (SAR)**: 4000.00

### Coffee
- **Transactions**: 
  - STARBUCKS COFFEE JEDDAH: -22.00
  - BARNS COFFEE: -19.00
  - DR CAFE COFFEE: -24.00
  - STARBUCKS COFFEE JEDDAH: -21.00
  - BARNS COFFEE: -18.50
  - STARBUCKS COFFEE JEDDAH: -23.00
  - DR CAFE COFFEE: -20.00
  - STARBUCKS COFFEE JEDDAH: -25.00
  - BARNS COFFEE: -19.50
  - DR CAFE COFFEE: -22.50
  - STARBUCKS COFFEE JEDDAH: -24.00
  - STARBUCKS COFFEE JEDDAH: -26.00
  - BARNS COFFEE: -18.00
  - STARBUCKS COFFEE JEDDAH: -22.50
  - STARBUCKS COFFEE JEDDAH: -23.50
  - DR CAFE COFFEE: -20.50
  - STARBUCKS COFFEE JEDDAH: -22.00
- **Count**: 17
- **Total Amount (SAR)**: -333.50

### Groceries
- **Transactions**: 
  - PANDA HYPERMARKET: -186.50
  - PANDA HYPERMARKET: -210.00
  - CARREFOUR MARKET: -142.00
  - PANDA HYPERMARKET: -175.00
  - PANDA H

In [ ]:
import urllib.request

logo_url = "https://raw.githubusercontent.com/ramahIO/Qirshak/main/data/qirshak_logo_base64.txt"
urllib.request.urlretrieve(logo_url, "qirshak_logo_base64.txt")

with open("qirshak_logo_base64.txt") as f:
    QIRSHAK_LOGO_B64 = f.read().strip()

print("Logo loaded:", len(QIRSHAK_LOGO_B64), "characters")

Logo loaded: 24304 characters


In [ ]:
# ============================================================
# Qirshak Interactive Gradio Interface (نسخة نهائية نظيفة)
# ============================================================

!pip install -q gradio

import gradio as gr
import urllib.request

# ---------- 🎨 ألوان الشعار الفعلية ----------
COLORS = {
    "primary": "#1C2B4A",        # كحلي (نفس النص بالشعار)
    "primary_light": "#EDE4CE",
    "accent": "#D4A72C",         # ذهبي العملة
    "background": "#F2EBDD",     # كريمي ورقي (نفس خلفية الشعار)
    "card_bg": "#FFFCF5",
}

CUSTOM_CSS = f"""
@import url('https://fonts.googleapis.com/css2?family=Tajawal:wght@400;500;700;800&display=swap');

/* فرض متغيرات Gradio الداخلية — يحل مشكلة النص الأبيض من الجذر */
:root, .dark {{
    --body-text-color: {COLORS['primary']} !important;
    --body-text-color-subdued: {COLORS['primary']} !important;
    --body-background-fill: {COLORS['background']} !important;
    --background-fill-primary: {COLORS['background']} !important;
    --background-fill-secondary: {COLORS['card_bg']} !important;
    --border-color-primary: {COLORS['accent']} !important;
    --block-title-text-color: {COLORS['primary']} !important;
    --block-label-text-color: {COLORS['primary']} !important;
    --button-primary-background-fill: {COLORS['accent']} !important;
    --button-primary-text-color: {COLORS['primary']} !important;
    --button-primary-background-fill-hover: {COLORS['primary']} !important;
    --button-primary-text-color-hover: {COLORS['accent']} !important;
    --neutral-700: {COLORS['primary']} !important;
    --neutral-800: {COLORS['primary']} !important;
}}

* {{
    font-family: 'Tajawal', 'Tahoma', sans-serif !important;
}}

.gradio-container {{
    background: {COLORS['background']} !important;
}}

#qirshak-header h1 {{
    color: {COLORS['primary']} !important;
    font-size: 36px !important;
    font-weight: 800 !important;
    margin: 12px 0 4px 0 !important;
}}
#qirshak-header p {{
    color: {COLORS['primary']} !important;
    opacity: 0.85;
    font-size: 16px !important;
}}

.qirshak-card, .qirshak-card * {{
    direction: rtl;
    text-align: right;
    color: {COLORS['primary']} !important;
}}
.qirshak-card {{
    border-radius: 16px;
    padding: 22px;
    font-size: 17px;
    line-height: 1.9;
}}

label, span, p {{
    color: {COLORS['primary']} !important;
}}
"""

# ---------- تحميل الشعار تلقائياً من الريبو ----------
logo_url = "https://raw.githubusercontent.com/ramahIO/Qirshak/main/data/qirshak_logo_base64.txt"
urllib.request.urlretrieve(logo_url, "qirshak_logo_base64.txt")

with open("qirshak_logo_base64.txt") as f:
    QIRSHAK_LOGO_B64 = f.read().strip()

print("Logo loaded:", len(QIRSHAK_LOGO_B64), "characters")


# ---------- التقرير الشهري النهائي ----------
try:
    monthly_report_html = f"""
    <div class="qirshak-card" style="background:{COLORS['card_bg']};
                border:2px solid {COLORS['accent']};">
        {final_qirshak_message.replace(chr(10), "<br>")}
    </div>
    """
except NameError:
    monthly_report_html = "<p>شغّلي الفريق كامل أول (Section 6) عشان يطلع التقرير.</p>"


# ---------- بناء الواجهة ----------
with gr.Blocks(title="قِرْشَك") as demo:
    gr.HTML(f"""
        <div id="qirshak-header" style="text-align:center; padding: 25px 0;">
            <img src="data:image/jpeg;base64,{QIRSHAK_LOGO_B64}"
                 style="width:110px; height:110px; border-radius:50%;
                        border: 3px solid {COLORS['primary']}; object-fit:cover;
                        box-shadow: 0 3px 10px rgba(0,0,0,0.15);" />
            <h1>قِرْشَك</h1>
            <p>رفيقك يحسب لك قبل لا تصرفين</p>
        </div>
    """)

    gr.HTML(monthly_report_html)

demo.launch(debug=True, css=CUSTOM_CSS)

Logo loaded: 24304 characters
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7e1238e25c855ce9df.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
